# Multivariate Evidence: Co-occurrence, Priority, Entropy, Reproducibility, Learnability

Self-contained: all analysis code is defined in this notebook (no external module). It trains no model and calls no provider; every number comes from archived files.

The notebook backs the results on the six evidence ratings, in the order of the manuscript's questions:

1. **Quality control** of the forced-choice label (agreement with released labels, with two annotators, across four LLMs, with intervals from the four-LLM audit records; and a test-retest check of the protocol, gpt-4o-mini rerun on the 100 audit posts under the rubric prompt at temperature 0).
2. **Profile**: how often evidence for two or more dimensions co-occurs.
3. **Rule**: what the forced-choice label does with co-occurring evidence (by dimension, and by pair and strength), and what that explains about disagreement, against a base rate; in the four-LLM audit, the same priority under each of the three prompts.
4. **Signal**: whether score-vector entropy indicates co-occurring evidence, with protocol flags and with annotator-coded ratings; sensitivity with only clear ratings counted and with text length as a covariate.
5. **Trust**: whether annotators and other LLMs reproduce the ratings, and how often each rater codes each dimension.
6. **Reuse**: whether the six-head models learn the ratings, against an always-NONE reference and against annotators.
7. **Supplement**: entropy and disagreement on the forced-choice label.

Conventions: cleaned-statement clusters for resampling (each blank statement is its own cluster) and for the five text-group folds; tie-preserving entropy strata; seed 2025. Run all cells in Colab. The notebook displays aggregate tables only; it does not display post text.

In [1]:
from pathlib import Path
import json
import os

import pandas as pd
from IPython.display import display

try:
    from google.colab import drive
    drive.mount('/content/drive')
    MENTAL_HEALTH_ROOT = Path('/content/drive/MyDrive/NLP_Projects/02_MentalHealth')
    ANALYSIS_DIR = MENTAL_HEALTH_ROOT / '04_AMPPS/01_multivariate_evidence'
except ImportError:
    ANALYSIS_DIR = Path.cwd().resolve()
    MENTAL_HEALTH_ROOT = Path(os.environ.get('MH_ROOT', ANALYSIS_DIR.parents[1]))

R2_ROOT = MENTAL_HEALTH_ROOT / '03_R2'
LABELS_CSV = R2_ROOT / '02_mini4oLabel/data/mental_health_unified_labels_final.csv'
ASPECT_PREDICTIONS_CSV = R2_ROOT / '06_AspectLabel/outputs/test_predictions_aspect.csv'
HUMAN_AUDIT_DIR = MENTAL_HEALTH_ROOT / '02_R1/01_human_validation_300'
HUMAN_AGREEMENT_CSV = R2_ROOT / '07_ValidationRobustness/04_human_aspect_agreement/outputs/human_validation_aspect_agreement_kappa_overall.csv'
MULTILLM_OUTPUT = R2_ROOT / '07_ValidationRobustness/02_multillm_protocol_sensitivity/outputs'
# The 21,600 four-LLM audit records, mh_labeling_final.csv, from the public OSF deposit https://doi.org/10.17605/OSF.IO/K9CAJ
MULTILLM_RECORDS_CSV = MENTAL_HEALTH_ROOT / '04_AMPPS/data/mh_labeling_final.csv'
OUTPUT = ANALYSIS_DIR / 'outputs'

required = [
    LABELS_CSV,
    ASPECT_PREDICTIONS_CSV,
    HUMAN_AUDIT_DIR / 'generated_subset/human_validation_sample_key_300.csv',
    HUMAN_AUDIT_DIR / 'outputs/human_validation_annotator1_300.xlsm',
    HUMAN_AUDIT_DIR / 'outputs/human_validation_annotator2_300.xlsm',
    HUMAN_AGREEMENT_CSV,
    MULTILLM_OUTPUT / 'hard_label_pairwise_robustness.csv',
    MULTILLM_OUTPUT / 'aspect_level_agreement_robustness.csv',
    MULTILLM_OUTPUT / 'aspect_count_pairwise_robustness.csv',
    MULTILLM_OUTPUT / 'cooccurrence_pattern_pairwise_robustness.csv',
    MULTILLM_OUTPUT / 'aspect_prevalence_by_model.csv',
    MULTILLM_OUTPUT / 'aspect_strength_both_present_agreement.csv',
    MULTILLM_RECORDS_CSV,
]
for path in required:
    if not path.exists():
        raise FileNotFoundError(path)
print(f'Inputs: found ({len(required)} files)')
# Show full tables in retained outputs (no pandas column truncation)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 250)
pd.set_option('display.max_colwidth', None)

Inputs: found (13 files)


In [2]:
"""Multivariate evidence analyses for the rubric-guided annotation protocol.

One implementation for the manuscript-facing results on the six evidence
ratings: how often evidence co-occurs, what the forced-choice label does with
co-occurring evidence, whether score-vector entropy indicates it, whether
independent raters reproduce it, and whether supervised models learn it. A
concise quality-control block on the forced-choice label comes first, and the
forced-choice entropy analyses for the supplement come last.

The analysis reads archived inputs only: the unified label file, the two
human-audit workbooks, the retained aspect-model test predictions, and the
summary tables written by the human-aspect-agreement and multi-LLM notebooks,
and the 21,600 four-LLM audit records.
It trains no model and calls no provider. It writes aggregate tables and never
writes post text.

Conventions follow the canonical entropy notebook: exact row normalization,
0*log(0)=0, tie-preserving entropy strata, and the cleaned statement as the
cluster for resampling (each blank statement forming its own cluster) and for
the five text-group folds.
"""

from __future__ import annotations

import hashlib
import itertools
import json
import math
import platform
import re
import sys
import warnings
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import scipy
import sklearn
import statsmodels
import statsmodels.formula.api as smf
from scipy import stats
from sklearn.metrics import roc_auc_score


NOTEBOOK_NAME = "MentalHealth_multivariate_evidence.ipynb"

EXPECTED_N = 53_043
N_CLASSES = 7
N_FOLDS = 5
ASPECTS = (
    "depression",
    "anxiety",
    "suicidal",
    "stress",
    "bipolar",
    "personality_disorder",
)
ASPECT_PRESENT_COLS = tuple(f"u_{aspect}_present" for aspect in ASPECTS)
ASPECT_STRENGTH_COLS = tuple(f"u_{aspect}_strength" for aspect in ASPECTS)
PROB_COLS = (
    "u_p_normal",
    "u_p_depression",
    "u_p_anxiety",
    "u_p_suicidal",
    "u_p_stress",
    "u_p_bipolar",
    "u_p_personality_disorder",
)
LABEL_ORDER = (
    "NORMAL",
    "DEPRESSION",
    "SUICIDAL",
    "ANXIETY",
    "STRESS",
    "BIPOLAR",
    "PERSONALITY_DISORDER",
)
EXPECTED_LABELS = set(LABEL_ORDER)
LEVEL_TO_INT = {"none": 0, "weak": 1, "clear": 2}
METRIC_NAMES = (
    "three_level_exact_agreement",
    "three_level_nominal_kappa",
    "three_level_linear_weighted_kappa",
    "binary_exact_agreement",
    "binary_nominal_kappa",
)
BOOTSTRAP_REPLICATES = 2_000        # cluster and post bootstraps
BOOTSTRAP_SEED = 2025
F1_BOOTSTRAP_REPLICATES = 1_000     # shared-post bootstrap, as in the aspect mean-F1 notebook
AUC_BOOTSTRAP_REPLICATES = 1_000


# --- canonical helpers, identical to the entropy-routing and human-aspect-agreement notebooks ---

def sha256_file(path: Path) -> str:
    """Return a streaming SHA-256 digest for *path*."""
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def clean_text(text: object) -> str:
    """Match the grouped-split statement normalization used in R2."""
    if pd.isna(text):
        return ""
    value = str(text).lower()
    value = re.sub(r"http\S+", " urltoken ", value)
    value = re.sub(r"@\w+", " usertoken ", value)
    value = re.sub(r"#(\w+)", r" hashtag_\1 ", value)
    value = re.sub(r"[^\w\s]", "", value)
    value = re.sub(r"\s+", " ", value)
    return value.strip()


def text_fold(text: str, n_folds: int = N_FOLDS) -> int:
    """Map a cleaned statement deterministically to one of *n_folds*."""
    digest = hashlib.blake2b(text.encode("utf-8"), digest_size=8).digest()
    return int.from_bytes(digest, byteorder="little", signed=False) % n_folds


def normalize_label(series: pd.Series) -> pd.Series:
    """Normalize released and protocol hard-label spellings."""
    return (
        series.astype(str)
        .str.strip()
        .str.upper()
        .str.replace(r"[\s-]+", "_", regex=True)
    )


def normalize_score_rows(scores: np.ndarray) -> np.ndarray:
    """Validate and row-normalize nonnegative class-score vectors."""
    values = np.asarray(scores, dtype=float)
    if values.ndim != 2:
        raise ValueError(f"Score matrix must be two-dimensional; got {values.ndim}D")
    if values.shape[1] != N_CLASSES:
        raise ValueError(
            f"Expected {N_CLASSES} score columns; found {values.shape[1]}"
        )
    if not np.isfinite(values).all():
        raise ValueError("Score vectors contain non-finite values")
    if (values < 0).any():
        raise ValueError("Score vectors contain negative values")
    row_sums = values.sum(axis=1)
    if (row_sums <= 0).any():
        bad_n = int((row_sums <= 0).sum())
        raise ValueError(f"Found {bad_n} score vectors with non-positive row sums")
    normalized = values / row_sums[:, None]
    if not np.allclose(normalized.sum(axis=1), 1.0, rtol=0.0, atol=1e-12):
        raise AssertionError("Row-normalized score vectors do not sum to one")
    return normalized


def shannon_entropy(scores: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Return normalized rows, entropy in nats, and normalized entropy.

    Zeros are not clipped.  Terms with probability zero are assigned exactly
    zero, implementing the limiting convention ``0 * log(0) = 0``.
    """
    probs = normalize_score_rows(scores)
    terms = np.zeros_like(probs)
    positive = probs > 0
    terms[positive] = -probs[positive] * np.log(probs[positive])
    entropy_nats = terms.sum(axis=1)
    entropy_normalized = entropy_nats / math.log(probs.shape[1])
    if (entropy_nats < 0).any() or (entropy_nats > math.log(N_CLASSES) + 1e-12).any():
        raise AssertionError("Entropy fell outside its mathematical bounds")
    return probs, entropy_nats, entropy_normalized


def tie_preserving_entropy_strata(
    entropy_nats: np.ndarray,
) -> tuple[pd.Categorical, pd.DataFrame]:
    """Create exact-zero plus empirical positive-entropy strata.

    Positive-entropy tertile cutpoints are estimated from the positive subset.
    Because interval membership is defined by score thresholds, equal entropy
    values are never split across strata.  The resulting counts need not be
    equal and are not described as quintiles.
    """
    entropy = np.asarray(entropy_nats, dtype=float)
    if entropy.ndim != 1 or not np.isfinite(entropy).all() or (entropy < 0).any():
        raise ValueError("Entropy must be a finite, nonnegative vector")
    positive = entropy[entropy > 0]
    if len(positive) == 0:
        categories = pd.Categorical(
            np.repeat("H=0", len(entropy)), categories=["H=0"], ordered=True
        )
        cuts = pd.DataFrame(
            [{"stratum": "H=0", "lower_nats": 0.0, "upper_nats": 0.0}]
        )
        return categories, cuts

    empirical = np.quantile(positive, [1 / 3, 2 / 3], method="linear")
    cutpoints = np.unique(empirical)
    bounds = np.concatenate(([0.0], cutpoints, [np.inf]))
    positive_names = [
        "positive_low",
        "positive_mid" if len(cutpoints) == 2 else "positive_high",
        "positive_high",
    ][: len(bounds) - 1]
    labels = ["H=0", *positive_names]
    assigned = np.full(len(entropy), None, dtype=object)
    assigned[entropy == 0] = "H=0"
    for index, name in enumerate(positive_names):
        lower = bounds[index]
        upper = bounds[index + 1]
        mask = (entropy > lower) & (entropy <= upper)
        assigned[mask] = name
    if pd.isna(assigned).any():
        raise AssertionError("Some entropy values were not assigned to a stratum")
    categories = pd.Categorical(assigned, categories=labels, ordered=True)

    rows: list[dict[str, object]] = [
        {
            "stratum": "H=0",
            "lower_nats": 0.0,
            "lower_inclusive": True,
            "upper_nats": 0.0,
            "upper_inclusive": True,
            "definition": "entropy_nats == 0",
        }
    ]
    for index, name in enumerate(positive_names):
        lower = float(bounds[index])
        upper = float(bounds[index + 1])
        rows.append(
            {
                "stratum": name,
                "lower_nats": lower,
                "lower_inclusive": False,
                "upper_nats": upper if np.isfinite(upper) else np.nan,
                "upper_inclusive": True,
                "definition": (
                    f"{lower:.12g} < entropy_nats <= {upper:.12g}"
                    if np.isfinite(upper)
                    else f"entropy_nats > {lower:.12g}"
                ),
            }
        )
    return categories, pd.DataFrame(rows)


def _write_csv(table: pd.DataFrame, path: Path) -> None:
    table.to_csv(path, index=False, lineterminator="\n")


def _hash_outputs(output_dir: Path, included_names: Iterable[str]) -> pd.DataFrame:
    """Hash only outputs explicitly generated by the current analysis."""
    rows = []
    for name in sorted(set(included_names)):
        path = output_dir / name
        if not path.is_file():
            raise FileNotFoundError(f"Expected generated output is missing: {path}")
        rows.append(
            {
                "file": path.name,
                "bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
        )
    return pd.DataFrame(rows)


def normalize_levels(series: pd.Series, field_name: str) -> pd.Series:
    normalized = series.astype("string").str.strip().str.lower()
    invalid = normalized.isna() | ~normalized.isin(LEVEL_TO_INT)
    if invalid.any():
        values = sorted(normalized.loc[invalid].astype(str).unique().tolist())
        raise AssertionError(f"Invalid values in {field_name}: {values}")
    return normalized


def cohen_kappa(first: np.ndarray, second: np.ndarray, n_levels: int, *, linear: bool) -> float:
    first = np.asarray(first, dtype=np.int8)
    second = np.asarray(second, dtype=np.int8)
    if first.shape != second.shape or first.size == 0:
        raise ValueError("Kappa inputs must be nonempty arrays with identical shapes")

    confusion = np.zeros((n_levels, n_levels), dtype=np.float64)
    np.add.at(confusion, (first, second), 1.0)
    expected = np.outer(confusion.sum(axis=1), confusion.sum(axis=0)) / first.size
    if linear:
        positions = np.arange(n_levels, dtype=np.float64)
        weights = np.abs(positions[:, None] - positions[None, :]) / (n_levels - 1)
    else:
        weights = np.ones((n_levels, n_levels), dtype=np.float64) - np.eye(n_levels)

    observed_disagreement = float(np.sum(weights * confusion))
    expected_disagreement = float(np.sum(weights * expected))
    if np.isclose(expected_disagreement, 0.0):
        return float("nan")
    return 1.0 - observed_disagreement / expected_disagreement


def metrics(first: np.ndarray, second: np.ndarray) -> np.ndarray:
    first = np.asarray(first, dtype=np.int8)
    second = np.asarray(second, dtype=np.int8)
    first_binary = (first > 0).astype(np.int8)
    second_binary = (second > 0).astype(np.int8)
    return np.asarray(
        [
            np.mean(first == second),
            cohen_kappa(first, second, 3, linear=False),
            cohen_kappa(first, second, 3, linear=True),
            np.mean(first_binary == second_binary),
            cohen_kappa(first_binary, second_binary, 2, linear=False),
        ],
        dtype=np.float64,
    )


# --------------------------------------------------------------------------
# Loading
# --------------------------------------------------------------------------

def cluster_bootstrap_statistic(
    cluster: np.ndarray,
    columns: np.ndarray,
    statistic,
    replicates: int = BOOTSTRAP_REPLICATES,
    seed: int = BOOTSTRAP_SEED,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """``statistic`` of the column sums, with percentile intervals from the same cleaned-statement cluster
    resamples as ``cluster_bootstrap_shares`` (multinomial cluster weights, same seed)."""
    cluster = np.asarray(cluster, dtype=int)
    columns = np.asarray(columns, dtype=float)
    n_clusters = int(cluster.max()) + 1
    sums = np.zeros((n_clusters, columns.shape[1]))
    np.add.at(sums, cluster, columns)
    with np.errstate(invalid="ignore", divide="ignore"):
        point = np.asarray(statistic(sums.sum(axis=0)), dtype=float)
        rng = np.random.default_rng(seed)
        probabilities = np.full(n_clusters, 1.0 / n_clusters)
        draws = np.empty((replicates, point.size))
        for replicate in range(replicates):
            weights = rng.multinomial(n_clusters, probabilities).astype(float)
            draws[replicate] = statistic(np.einsum("g,gk->k", weights, sums))
    low, high = np.nanpercentile(draws, [2.5, 97.5], axis=0)
    return point, low, high


def load_corpus(path: Path) -> tuple[pd.DataFrame, np.ndarray, pd.DataFrame]:
    """Read the unified label file and derive every row-level quantity once."""
    columns = ["statement", "status", "u_label", *PROB_COLS, *ASPECT_PRESENT_COLS, *ASPECT_STRENGTH_COLS]
    df = pd.read_csv(path, usecols=columns, low_memory=False).reset_index(drop=True)
    if len(df) != EXPECTED_N:
        raise ValueError(f"Expected {EXPECTED_N:,} records, found {len(df):,}")
    df["released_label"] = normalize_label(df["status"])
    df["protocol_label"] = normalize_label(df["u_label"])
    unexpected = (set(df["released_label"]) | set(df["protocol_label"])) - EXPECTED_LABELS
    if unexpected:
        raise ValueError(f"Unexpected normalized labels: {sorted(unexpected)}")
    df["disagreement"] = (df["released_label"] != df["protocol_label"]).astype(int)
    flags = df.loc[:, list(ASPECT_PRESENT_COLS)].astype(bool).to_numpy()
    df["n_dimensions"] = flags.sum(axis=1).astype(int)
    _probs, entropy_nats, entropy_normalized = shannon_entropy(
        df.loc[:, list(PROB_COLS)].to_numpy(dtype=float)
    )
    df["entropy_nats"] = entropy_nats
    df["normalized_entropy"] = entropy_normalized
    strata, cutpoints = tie_preserving_entropy_strata(entropy_nats)
    df["entropy_stratum"] = strata
    clean = df["statement"].map(clean_text)
    # Posts whose cleaned statement is blank share no text, so each forms its own cluster.
    cluster = pd.factorize(clean.mask(clean == ""))[0]
    blank = cluster < 0
    cluster[blank] = cluster.max() + 1 + np.arange(int(blank.sum()))
    df["cluster"] = cluster
    df["group_fold"] = clean.map(text_fold)
    df["n_words"] = clean.str.split().str.len().astype(int)
    df["text_md5"] = df["statement"].map(lambda text: hashlib.md5(str(text).encode()).hexdigest())  # no text kept
    df["blank_statement"] = df["statement"].fillna("").astype(str).str.strip().eq("")
    for aspect, column in zip(ASPECTS, ASPECT_STRENGTH_COLS):
        df[f"level_{aspect}"] = normalize_levels(df[column], column).map(LEVEL_TO_INT).astype(int)
    df = df.drop(columns=["statement", "status", "u_label", *ASPECT_STRENGTH_COLS])  # no post text beyond this point
    return df, flags, cutpoints


def canonical_label(value: object) -> str:
    """Normalize annotator and key spellings of the seven primary labels."""
    if pd.isna(value):
        return ""
    text = re.sub(r"_+", "_", str(value).strip().upper().replace("-", "_").replace(" ", "_"))
    aliases = {
        "PERSONALITY": "PERSONALITY_DISORDER",
        "PERSONALITY_DIS": "PERSONALITY_DISORDER",
        "PERSONALITY_DIS.": "PERSONALITY_DISORDER",
        "PERSONALITYDISORDER": "PERSONALITY_DISORDER",
        "PERS_DIS": "PERSONALITY_DISORDER",
        "PERS_DIS.": "PERSONALITY_DISORDER",
        "PERS._DIS.": "PERSONALITY_DISORDER",
        "NONE": "NORMAL",
    }
    return aliases.get(text, text)


def read_audit(audit_dir: Path) -> pd.DataFrame:
    """Merge the sample key with both annotator workbooks; one row per audited post."""
    key = pd.read_csv(audit_dir / "generated_subset" / "human_validation_sample_key_300.csv")
    frames = {}
    for k in (1, 2):
        raw = pd.read_excel(
            audit_dir / "outputs" / f"human_validation_annotator{k}_300.xlsm", sheet_name="Annotation"
        )
        raw = raw.loc[raw["sample_id"].notna()].copy()
        raw["sample_id"] = raw["sample_id"].astype(str).str.strip()
        frames[k] = raw
    for name, frame in (("annotator_1", frames[1]), ("annotator_2", frames[2]), ("sample_key", key)):
        if len(frame) != 300 or frame["sample_id"].nunique() != 300:
            raise AssertionError(f"{name} does not contain 300 unique sample IDs")
    key["sample_id"] = key["sample_id"].astype(str).str.strip()
    if set(frames[1]["sample_id"]) != set(key["sample_id"]) or set(frames[2]["sample_id"]) != set(key["sample_id"]):
        raise AssertionError("The three audit inputs do not contain identical sample-ID sets")

    out = pd.DataFrame({"sample_id": key["sample_id"]})
    out["released_label"] = key["status"].map(canonical_label)
    out["protocol_label"] = key["u_label"].map(canonical_label)
    _p, entropy_nats, _n = shannon_entropy(key.loc[:, list(PROB_COLS)].to_numpy(dtype=float))
    out["entropy_nats"] = entropy_nats
    out["clean_statement"] = key["statement"].map(clean_text)
    out["n_words"] = out["clean_statement"].str.split().str.len().astype(int)
    for aspect in ASPECTS:
        levels = normalize_levels(key[f"u_{aspect}_strength"], f"u_{aspect}_strength")
        out[f"protocol_{aspect}"] = levels.map(LEVEL_TO_INT).astype(int)
    for k in (1, 2):
        part = pd.DataFrame({"sample_id": frames[k]["sample_id"]})
        part[f"a{k}_label"] = frames[k]["human_primary_label"].map(canonical_label)
        for aspect in ASPECTS:
            levels = normalize_levels(frames[k][f"human_{aspect}"], f"annotator_{k}.human_{aspect}")
            part[f"a{k}_{aspect}"] = levels.map(LEVEL_TO_INT).astype(int).to_numpy()
        out = out.merge(part, on="sample_id", how="inner", validate="one_to_one")
    if len(out) != 300:
        raise AssertionError("Audit merge did not return 300 posts")
    for column in ("released_label", "protocol_label", "a1_label", "a2_label"):
        bad = set(out[column]) - EXPECTED_LABELS
        if bad:
            raise AssertionError(f"Unexpected labels in {column}: {sorted(bad)}")
    for rater in ("protocol", "a1", "a2"):
        out[f"{rater}_n_dimensions"] = sum((out[f"{rater}_{aspect}"] > 0).astype(int) for aspect in ASPECTS)
    return out


# --------------------------------------------------------------------------
# Resampling helpers
# --------------------------------------------------------------------------

def cluster_bootstrap_shares(
    cluster: np.ndarray,
    numerators: np.ndarray,
    denominators: np.ndarray,
    replicates: int = BOOTSTRAP_REPLICATES,
    seed: int = BOOTSTRAP_SEED,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Shares with percentile intervals from a cleaned-statement cluster bootstrap.

    ``numerators`` and ``denominators`` are row-by-quantity indicator arrays.
    Clusters are resampled with replacement through multinomial weights, so
    every quantity in one call shares the same resamples.
    """
    cluster = np.asarray(cluster, dtype=int)
    numerators = np.asarray(numerators, dtype=float)
    denominators = np.asarray(denominators, dtype=float)
    if numerators.ndim == 1:
        numerators = numerators[:, None]
    if denominators.ndim == 1:
        denominators = denominators[:, None]
    n_clusters = int(cluster.max()) + 1
    num = np.zeros((n_clusters, numerators.shape[1]))
    den = np.zeros((n_clusters, denominators.shape[1]))
    np.add.at(num, cluster, numerators)
    np.add.at(den, cluster, denominators)
    with np.errstate(invalid="ignore", divide="ignore"):
        point = num.sum(axis=0) / den.sum(axis=0)
        rng = np.random.default_rng(seed)
        probabilities = np.full(n_clusters, 1.0 / n_clusters)
        draws = np.empty((replicates, numerators.shape[1]))
        for replicate in range(replicates):
            weights = rng.multinomial(n_clusters, probabilities).astype(float)
            # einsum keeps the weighted sums out of BLAS matmul, which raises spurious
            # floating-point warnings on some platforms
            draws[replicate] = np.einsum("g,gk->k", weights, num) / np.einsum("g,gk->k", weights, den)
    low, high = np.nanpercentile(draws, [2.5, 97.5], axis=0)
    return point, low, high


def post_bootstrap(statistic, n_posts: int, replicates: int = BOOTSTRAP_REPLICATES, seed: int = BOOTSTRAP_SEED):
    """Percentile interval for ``statistic(index)`` when posts are resampled with replacement."""
    rng = np.random.default_rng(seed)
    values = []
    for _ in range(replicates):
        index = rng.integers(0, n_posts, n_posts)
        values.append(statistic(index))
    values = np.asarray(values, dtype=float)
    return np.nanpercentile(values, [2.5, 97.5], axis=0)


def label_codes(series: pd.Series) -> np.ndarray:
    return series.map({name: i for i, name in enumerate(LABEL_ORDER)}).to_numpy(dtype=np.int8)


# --------------------------------------------------------------------------
# Quality control: the forced-choice label
# --------------------------------------------------------------------------

def quality_control_table(df: pd.DataFrame, audit: pd.DataFrame, multillm_dir: Path, records: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    released = label_codes(df["released_label"])
    protocol = label_codes(df["protocol_label"])
    kappa = cohen_kappa(released, protocol, len(LABEL_ORDER), linear=False)
    # Cleaned-statement cluster bootstrap, as for the other full-corpus intervals: the confusion counts of each
    # resample are the cluster-weighted sums of one-hot (released, protocol) cells.
    levels = len(LABEL_ORDER)
    cells = np.zeros((len(df), levels * levels))
    cells[np.arange(len(df)), released.astype(int) * levels + protocol.astype(int)] = 1.0

    def kappa_from_counts(counts):
        table = counts.reshape(levels, levels)
        total = table.sum()
        observed, expected = np.trace(table) / total, table.sum(axis=1) @ table.sum(axis=0) / total ** 2
        return np.array([(observed - expected) / (1 - expected)])

    point_check, low, high = cluster_bootstrap_statistic(df["cluster"].to_numpy(), cells, kappa_from_counts)
    if abs(float(point_check[0]) - kappa) > 1e-12:
        raise AssertionError("Cluster-bootstrap kappa does not reproduce the point estimate")
    low, high = float(low[0]), float(high[0])
    rows.append({
        "check": "released labels", "comparison": "Protocol vs. released label", "sample": "full corpus",
        "n": len(df), "n_agree": int((released == protocol).sum()), "agreement": float(np.mean(released == protocol)),
        "kappa": kappa, "kappa_ci_low": float(low), "kappa_ci_high": float(high),
        "interval": f"cleaned-statement cluster bootstrap, {BOOTSTRAP_REPLICATES} resamples",
    })
    pairs = (
        ("human coders", "Annotator 1 vs. Annotator 2", "a1_label", "a2_label"),
        ("human coders", "Annotator 1 vs. protocol", "a1_label", "protocol_label"),
        ("human coders", "Annotator 2 vs. protocol", "a2_label", "protocol_label"),
        ("human coders", "Annotator 1 vs. released label", "a1_label", "released_label"),
        ("human coders", "Annotator 2 vs. released label", "a2_label", "released_label"),
        ("human coders", "Protocol vs. released label", "protocol_label", "released_label"),
    )
    for check, name, left, right in pairs:
        first, second = label_codes(audit[left]), label_codes(audit[right])
        ci = post_bootstrap(lambda idx: cohen_kappa(first[idx], second[idx], len(LABEL_ORDER), linear=False), len(audit))
        rows.append({
            "check": check, "comparison": name, "sample": "human audit (enriched)", "n": len(audit),
            "n_agree": int((first == second).sum()), "agreement": float(np.mean(first == second)),
            "kappa": cohen_kappa(first, second, len(LABEL_ORDER), linear=False),
            "kappa_ci_low": float(ci[0]), "kappa_ci_high": float(ci[1]),
            "interval": f"post bootstrap, {BOOTSTRAP_REPLICATES} resamples",
        })
    hard = pd.read_csv(multillm_dir / "hard_label_pairwise_robustness.csv")
    fixed = hard.loc[hard["scope"] == "rubric_t0_seed1001"]
    if len(fixed) != 6:
        raise AssertionError("Expected six fixed-condition LLM pairs")
    # Intervals from the audit records: the multi-LLM notebook's labels must reproduce its agreement and kappa exactly.
    condition = records[(records["prompt_type"] == "rubric") & (records["temperature"] == 0) & (records["seed"] == 1001)]
    wide = condition.pivot(index="item_id", columns="evaluator", values="label").sort_index()
    for _, pair in fixed.iterrows():
        first, second = label_codes(wide[pair["model_a"]]), label_codes(wide[pair["model_b"]])
        kappa = cohen_kappa(first, second, len(LABEL_ORDER), linear=False)
        if abs(kappa - float(pair["cohen_kappa_nominal"])) > 1e-12 or float(np.mean(first == second)) != float(pair["agreement"]):
            raise AssertionError(f"Audit records do not reproduce {pair['model_a']} vs. {pair['model_b']}")
        low, high = post_bootstrap(lambda idx: cohen_kappa(first[idx], second[idx], len(LABEL_ORDER), linear=False), len(wide))
        rows.append({
            "check": "other LLMs", "comparison": f"{pair['model_a']} vs. {pair['model_b']}",
            "sample": "multi-LLM audit, rubric prompt, temperature 0, seed 1001", "n": int(pair["n_matched_outputs"]),
            "n_agree": int(round(pair["agreement"] * pair["n_matched_outputs"])), "agreement": float(pair["agreement"]),
            "kappa": float(pair["cohen_kappa_nominal"]), "kappa_ci_low": float(low), "kappa_ci_high": float(high),
            "interval": f"post bootstrap, {BOOTSTRAP_REPLICATES} resamples",
        })
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------
# Profile: how often evidence co-occurs
# --------------------------------------------------------------------------

def profile_tables(df: pd.DataFrame, flags: np.ndarray) -> tuple[pd.DataFrame, pd.DataFrame]:
    k = df["n_dimensions"].to_numpy()
    groups = (("0", k == 0), ("1", k == 1), ("2", k == 2), ("3", k == 3), ("4 or more", k >= 4), ("2 or more", k >= 2))
    numerators = np.column_stack([mask for _name, mask in groups]).astype(float)
    point, low, high = cluster_bootstrap_shares(df["cluster"].to_numpy(), numerators, np.ones_like(numerators))
    counts = pd.DataFrame({
        "n_dimensions": [name for name, _mask in groups],
        "n_posts": [int(mask.sum()) for _name, mask in groups],
        "share": point, "share_ci_low": low, "share_ci_high": high,
    })
    rows = []
    for i in range(len(ASPECTS)):
        for j in range(i + 1, len(ASPECTS)):
            both = flags[:, i] & flags[:, j]
            rows.append({
                "dimension_a": ASPECTS[i], "dimension_b": ASPECTS[j],
                "n_posts_both": int(both.sum()), "share_of_corpus": float(both.mean()),
            })
    pairs = pd.DataFrame(rows).sort_values("n_posts_both", ascending=False).reset_index(drop=True)
    return counts, pairs


# --------------------------------------------------------------------------
# Rule: what the forced-choice label does with co-occurring evidence
# --------------------------------------------------------------------------

def rule_tables(df: pd.DataFrame, flags: np.ndarray) -> dict[str, pd.DataFrame]:
    cluster = df["cluster"].to_numpy()
    k = df["n_dimensions"].to_numpy()
    released = df["released_label"].to_numpy()
    protocol = df["protocol_label"].to_numpy()
    disagree = df["disagreement"].to_numpy().astype(bool)
    multi = k >= 2

    # Priority shares: among posts with two or more flagged dimensions that include d,
    # how often the forced-choice label is d.
    numerators, denominators = [], []
    for i, aspect in enumerate(ASPECTS):
        eligible = multi & flags[:, i]
        numerators.append(eligible & (protocol == aspect.upper()))
        denominators.append(eligible)
    point, low, high = cluster_bootstrap_shares(cluster, np.column_stack(numerators), np.column_stack(denominators))
    priority = pd.DataFrame({
        "dimension": ASPECTS,
        "n_posts_with_dimension_among_two_or_more": [int(mask.sum()) for mask in denominators],
        "share_forced_choice_is_dimension": point, "share_ci_low": low, "share_ci_high": high,
    }).sort_values("share_forced_choice_is_dimension", ascending=False).reset_index(drop=True)

    # Disagreement by released category and number of flagged dimensions.
    dimension_groups = (("0", k == 0), ("1", k == 1), ("2 or more", k >= 2))
    cells, numerators, denominators = [], [], []
    for label in [*LABEL_ORDER, "ALL"]:
        in_label = np.ones(len(df), dtype=bool) if label == "ALL" else (released == label)
        for name, mask in dimension_groups:
            cell = in_label & mask
            cells.append((label, name, int(cell.sum())))
            numerators.append(cell & disagree)
            denominators.append(cell)
    point, low, high = cluster_bootstrap_shares(cluster, np.column_stack(numerators), np.column_stack(denominators))
    by_category = pd.DataFrame({
        "released_label": [c[0] for c in cells], "n_dimensions": [c[1] for c in cells], "n_posts": [c[2] for c in cells],
        "disagreement_rate": point, "rate_ci_low": low, "rate_ci_high": high,
    })
    by_category.loc[by_category["n_posts"] == 0, ["disagreement_rate", "rate_ci_low", "rate_ci_high"]] = np.nan

    # The largest co-occurring pair, split by the released label.
    both = flags[:, ASPECTS.index("depression")] & flags[:, ASPECTS.index("suicidal")]
    groups = (("SUICIDAL", both & (released == "SUICIDAL")), ("DEPRESSION", both & (released == "DEPRESSION")), ("ALL", both))
    numerators = np.column_stack([np.column_stack([mask & disagree, mask & (protocol == "SUICIDAL")]) for _n, mask in groups])
    denominators = np.column_stack([np.column_stack([mask, mask]) for _n, mask in groups])
    point, low, high = cluster_bootstrap_shares(cluster, numerators, denominators)
    contrast = pd.DataFrame({
        "evidence": "depression and suicidal both flagged",
        "released_label": [name for name, _m in groups], "n_posts": [int(mask.sum()) for _n, mask in groups],
        "disagreement_rate": point[0::2], "rate_ci_low": low[0::2], "rate_ci_high": high[0::2],
        "share_forced_choice_suicidal": point[1::2], "share_ci_low": low[1::2], "share_ci_high": high[1::2],
    })

    # Four mutually exclusive types of disagreement.
    released_flagged = np.zeros(len(df), dtype=bool)
    for i, aspect in enumerate(ASPECTS):
        mask = released == aspect.upper()
        released_flagged[mask] = flags[mask, i]
    types = (
        ("Released label Normal, protocol label a category", disagree & (released == "NORMAL")),
        ("Protocol label Normal", disagree & (released != "NORMAL") & (protocol == "NORMAL")),
        ("Different category, released category also flagged (co-occurring evidence)",
         disagree & (released != "NORMAL") & (protocol != "NORMAL") & released_flagged),
        ("Different category, released category not flagged",
         disagree & (released != "NORMAL") & (protocol != "NORMAL") & ~released_flagged),
    )
    if sum(int(mask.sum()) for _n, mask in types) != int(disagree.sum()):
        raise AssertionError("Disagreement types do not partition the disagreements")
    numerators = np.column_stack([mask for _n, mask in types])
    denominators = np.column_stack([disagree for _n, _m in types])
    point, low, high = cluster_bootstrap_shares(cluster, numerators, denominators)
    type_table = pd.DataFrame({
        "disagreement_type": [name for name, _m in types], "n_posts": [int(mask.sum()) for _n, mask in types],
        "share_of_disagreements": point, "share_ci_low": low, "share_ci_high": high,
        "share_of_corpus": [float(mask.mean()) for _n, mask in types],
    })
    return {"priority": priority, "by_category": by_category, "contrast": contrast, "types": type_table}


# --------------------------------------------------------------------------
# Signal: score-vector entropy and co-occurring evidence
# --------------------------------------------------------------------------

def signal_tables(df: pd.DataFrame, audit: pd.DataFrame, cutpoints: pd.DataFrame) -> dict[str, pd.DataFrame]:
    cluster = df["cluster"].to_numpy()
    k = df["n_dimensions"].to_numpy()
    entropy = df["entropy_nats"].to_numpy()
    strata = df["entropy_stratum"].astype(str).to_numpy()
    stratum_names = [str(name) for name in df["entropy_stratum"].cat.categories]
    multi = k >= 2

    numerators = np.column_stack([(strata == name) & multi for name in stratum_names] + [multi])
    denominators = np.column_stack([strata == name for name in stratum_names] + [np.ones(len(df), dtype=bool)])
    point, low, high = cluster_bootstrap_shares(cluster, numerators, denominators)
    by_stratum = pd.DataFrame({
        "entropy_stratum": [*stratum_names, "ALL"],
        "n_posts": [int((strata == name).sum()) for name in stratum_names] + [len(df)],
        "share_two_or_more_dimensions": point, "share_ci_low": low, "share_ci_high": high,
        "mean_n_dimensions": [float(k[strata == name].mean()) for name in stratum_names] + [float(k.mean())],
    })

    by_count = pd.DataFrame([
        {"n_dimensions": name, "n_posts": int(mask.sum()), "mean_entropy_nats": float(entropy[mask].mean()),
         "share_zero_entropy": float((entropy[mask] == 0).mean())}
        for name, mask in (("0", k == 0), ("1", k == 1), ("2", k == 2), ("3", k == 3), ("4 or more", k >= 4), ("2 or more", k >= 2))
    ])

    # Audit posts placed in the same corpus strata; annotator-coded dimensions.
    upper = float(cutpoints.loc[cutpoints["stratum"] == "positive_mid", "upper_nats"].iloc[0])
    top = audit["entropy_nats"].to_numpy() > upper
    rows = []
    for rater, label in (("protocol", "Protocol flags"), ("a1", "Annotator 1"), ("a2", "Annotator 2")):
        coded = (audit[f"{rater}_n_dimensions"].to_numpy() >= 2).astype(float)
        for name, mask in (("top stratum", top), ("all other strata", ~top)):
            ci = post_bootstrap(lambda idx: coded[idx][mask[idx]].mean() if mask[idx].any() else np.nan, len(audit))
            rows.append({"rater": label, "audit_posts": name, "n_posts": int(mask.sum()),
                         "share_two_or_more_dimensions": float(coded[mask].mean()), "share_ci_low": ci[0], "share_ci_high": ci[1]})
        diff = post_bootstrap(lambda idx: coded[idx][top[idx]].mean() - coded[idx][~top[idx]].mean(), len(audit))
        rho, p_value = stats.spearmanr(audit["entropy_nats"], audit[f"{rater}_n_dimensions"])
        rows.append({"rater": label, "audit_posts": "difference (top minus other)", "n_posts": len(audit),
                     "share_two_or_more_dimensions": float(coded[top].mean() - coded[~top].mean()),
                     "share_ci_low": diff[0], "share_ci_high": diff[1],
                     "spearman_entropy_vs_n_dimensions": float(rho), "spearman_p_value": float(p_value)})
    annotators = pd.DataFrame(rows)

    # Supplementary indices for the same relation.
    with_dimension = k >= 1
    frame = pd.DataFrame({
        "multi": multi[with_dimension].astype(int),
        "entropy_z": (entropy[with_dimension] - entropy[with_dimension].mean()) / entropy[with_dimension].std(),
        "protocol_label": df.loc[with_dimension, "protocol_label"].to_numpy(),
    })
    adjusted = smf.logit("multi ~ entropy_z + C(protocol_label)", data=frame).fit(
        disp=0, cov_type="cluster", cov_kwds={"groups": cluster[with_dimension]}
    )
    beta, se = float(adjusted.params["entropy_z"]), float(adjusted.bse["entropy_z"])
    extra = pd.DataFrame([
        {"index": "AUC of entropy for two or more dimensions, all posts", "value": roc_auc_score(multi, entropy)},
        {"index": "AUC of entropy for two or more versus exactly one dimension", "value": roc_auc_score(multi[with_dimension], entropy[with_dimension])},
        {"index": "Spearman rho, entropy and number of dimensions, all posts", "value": stats.spearmanr(entropy, k)[0]},
        {"index": "Share of posts with two or more dimensions that have zero entropy", "value": float((entropy[multi] == 0).mean())},
        {"index": "Odds ratio for two or more dimensions per 1 SD entropy, adjusted for protocol label (posts with at least one dimension)",
         "value": math.exp(beta), "ci_low": math.exp(beta - 1.959964 * se), "ci_high": math.exp(beta + 1.959964 * se)},
    ])
    return {"by_stratum": by_stratum, "by_count": by_count, "annotators": annotators, "extra": extra}


# --------------------------------------------------------------------------
# Trust: independent raters
# --------------------------------------------------------------------------

def trust_tables(audit: pd.DataFrame, human_agreement_csv: Path, multillm_dir: Path) -> dict[str, pd.DataFrame]:
    # Per-dimension agreement: point estimates recomputed here and checked against
    # the human-aspect-agreement notebook, whose table carries the intervals.
    existing = pd.read_csv(human_agreement_csv)
    existing = existing.loc[existing["aggregation"] == "macro_across_six_aspects"].set_index("comparison")
    rows = []
    for name, left, right in (
        ("Annotator 1 vs. Annotator 2", "a1", "a2"),
        ("Annotator 1 vs. AI aspects", "a1", "protocol"),
        ("Annotator 2 vs. AI aspects", "a2", "protocol"),
    ):
        per_aspect = np.vstack([metrics(audit[f"{left}_{a}"].to_numpy(), audit[f"{right}_{a}"].to_numpy()) for a in ASPECTS])
        macro = np.nanmean(per_aspect, axis=0)
        row = {"comparison": name}
        for metric_name, value in zip(METRIC_NAMES, macro):
            row[metric_name] = float(value)
            reference = float(existing.loc[name, metric_name])
            if not np.isclose(value, reference, atol=5e-7):
                raise AssertionError(f"{name} {metric_name}: {value} differs from archived {reference}")
            row[f"{metric_name}_ci_low"] = float(existing.loc[name, f"{metric_name}_ci_low"])
            row[f"{metric_name}_ci_high"] = float(existing.loc[name, f"{metric_name}_ci_high"])
        rows.append(row)
    per_dimension = pd.DataFrame(rows)

    rows = []
    for name, left, right in (
        ("Annotator 1 vs. Annotator 2", "a1", "a2"),
        ("Annotator 1 vs. protocol", "a1", "protocol"),
        ("Annotator 2 vs. protocol", "a2", "protocol"),
    ):
        first = (audit[f"{left}_n_dimensions"].to_numpy() >= 2).astype(np.int8)
        second = (audit[f"{right}_n_dimensions"].to_numpy() >= 2).astype(np.int8)
        ci = post_bootstrap(lambda idx: cohen_kappa(first[idx], second[idx], 2, linear=False), len(audit))
        rho = stats.spearmanr(audit[f"{left}_n_dimensions"], audit[f"{right}_n_dimensions"])[0]
        rows.append({"comparison": name, "n_posts": len(audit), "exact_agreement_two_or_more": float(np.mean(first == second)),
                     "kappa_two_or_more": cohen_kappa(first, second, 2, linear=False), "kappa_ci_low": ci[0], "kappa_ci_high": ci[1],
                     "spearman_n_dimensions": float(rho)})
    two_or_more = pd.DataFrame(rows)

    # Disagreements between two non-Normal labels: did the rater code both categories as present?
    subset = audit.loc[(audit["released_label"] != audit["protocol_label"]) & (audit["released_label"] != "NORMAL")
                       & (audit["protocol_label"] != "NORMAL")].reset_index(drop=True)
    rows = []
    for rater, label in (("protocol", "Protocol flags"), ("a1", "Annotator 1"), ("a2", "Annotator 2")):
        released_present = np.array([row[f"{rater}_{row['released_label'].lower()}"] > 0 for _, row in subset.iterrows()])
        protocol_present = np.array([row[f"{rater}_{row['protocol_label'].lower()}"] > 0 for _, row in subset.iterrows()])
        both = released_present & protocol_present
        ci = post_bootstrap(lambda idx: both[idx].mean(), len(subset))
        rows.append({"rater": label, "n_disagreements_between_two_categories": len(subset),
                     "share_released_category_present": float(released_present.mean()),
                     "share_protocol_category_present": float(protocol_present.mean()),
                     "share_both_present": float(both.mean()), "both_ci_low": ci[0], "both_ci_high": ci[1]})
    overlap = pd.DataFrame(rows)

    agreement = pd.read_csv(multillm_dir / "aspect_level_agreement_robustness.csv")
    macro = agreement.loc[agreement["scope"] == "overall_macro_pair_aspect"]
    counts = pd.read_csv(multillm_dir / "aspect_count_pairwise_robustness.csv")
    patterns = pd.read_csv(multillm_dir / "cooccurrence_pattern_pairwise_robustness.csv")
    rows = []
    for _, row in macro.iterrows():
        rows.append({"quantity": f"per-dimension agreement, {row['representation']}, {row['inclusion_definition']}",
                     "exact_agreement": float(row["exact_agreement"]), "nominal_kappa": float(row["nominal_kappa"]),
                     "linear_weighted_kappa": float(row["linear_weighted_kappa"]) if pd.notna(row["linear_weighted_kappa"]) else np.nan})
    rows.append({"quantity": "number of dimensions per post, Spearman rho across six LLM pairs",
                 "minimum": float(counts["spearman_rho"].min()), "maximum": float(counts["spearman_rho"].max())})
    rows.append({"quantity": "co-occurrence pattern (fifteen pair rates), Spearman rho across six LLM pairs",
                 "minimum": float(patterns["spearman_rho"].min()), "maximum": float(patterns["spearman_rho"].max())})
    multillm = pd.DataFrame(rows)
    return {"per_dimension": per_dimension, "two_or_more": two_or_more, "overlap": overlap, "multillm": multillm}


# --------------------------------------------------------------------------
# Reuse: supervised models trained on the evidence ratings
# --------------------------------------------------------------------------

def f1_weighted_macro(truth: np.ndarray, predicted: np.ndarray, n_levels: int = 3) -> tuple[float, float, np.ndarray]:
    confusion = np.bincount(truth * n_levels + predicted, minlength=n_levels * n_levels).reshape(n_levels, n_levels).astype(float)
    support = confusion.sum(axis=1)
    denominator = support + confusion.sum(axis=0)
    f1 = np.divide(2.0 * np.diag(confusion), denominator, out=np.zeros(n_levels), where=denominator > 0)
    return float(np.sum(f1 * support) / support.sum()), float(f1.mean()), f1


def reuse_tables(prediction_csv: Path, audit: pd.DataFrame) -> dict[str, pd.DataFrame]:
    predictions = pd.read_csv(prediction_csv)
    predictions["clean_statement"] = predictions["statement"].map(clean_text)
    predictions = predictions.drop(columns=["statement"])
    raters = {"Always NONE": None, "ALBERT": "albert", "BioBERT": "biobert"}

    rows = []
    for aspect in ASPECTS:
        truth = predictions[f"{aspect}_true"].to_numpy(dtype=int)
        shares = np.bincount(truth, minlength=3) / len(truth)
        for label, prefix in raters.items():
            predicted = np.zeros_like(truth) if prefix is None else predictions[f"{prefix}_{aspect}_pred"].to_numpy(dtype=int)
            weighted, macro, per_class = f1_weighted_macro(truth, predicted)
            rows.append({"dimension": aspect, "rater": label, "n_test_posts": len(truth),
                         "share_none": shares[0], "share_weak": shares[1], "share_clear": shares[2],
                         "weighted_f1": weighted, "macro_f1": macro, "f1_none": per_class[0], "f1_weak": per_class[1], "f1_clear": per_class[2]})
    by_head = pd.DataFrame(rows)
    means = by_head.groupby("rater", sort=False)[["weighted_f1", "macro_f1", "f1_weak", "f1_clear"]].mean().reset_index()

    # Shared-post bootstrap of the six-head mean difference against the always-NONE reference.
    truth = np.column_stack([predictions[f"{a}_true"].to_numpy(dtype=int) for a in ASPECTS])
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    draws = {label: [] for label in ("ALBERT", "BioBERT")}
    model_predictions = {label: np.column_stack([predictions[f"{raters[label]}_{a}_pred"].to_numpy(dtype=int) for a in ASPECTS])
                         for label in ("ALBERT", "BioBERT")}
    for _ in range(F1_BOOTSTRAP_REPLICATES):
        index = rng.integers(0, len(truth), len(truth))
        reference = np.array([f1_weighted_macro(truth[index, j], np.zeros(len(index), dtype=int))[:2] for j in range(len(ASPECTS))]).mean(axis=0)
        for label in draws:
            model = np.array([f1_weighted_macro(truth[index, j], model_predictions[label][index, j])[:2] for j in range(len(ASPECTS))]).mean(axis=0)
            draws[label].append(model - reference)
    rows = []
    reference_row = means.loc[means["rater"] == "Always NONE"].iloc[0]
    for label in draws:
        values = np.asarray(draws[label])
        model_row = means.loc[means["rater"] == label].iloc[0]
        rows.append({"rater": label,
                     "mean_weighted_f1_minus_always_none": float(model_row["weighted_f1"] - reference_row["weighted_f1"]),
                     "weighted_ci_low": np.percentile(values[:, 0], 2.5), "weighted_ci_high": np.percentile(values[:, 0], 97.5),
                     "mean_macro_f1_minus_always_none": float(model_row["macro_f1"] - reference_row["macro_f1"]),
                     "macro_ci_low": np.percentile(values[:, 1], 2.5), "macro_ci_high": np.percentile(values[:, 1], 97.5)})
    differences = pd.DataFrame(rows)

    # Held-out audit posts: models, protocol ratings and annotators on the same posts.
    unique_predictions = predictions.drop_duplicates("clean_statement")
    held_out = audit.merge(unique_predictions, on="clean_statement", how="inner", validate="one_to_one")
    rows = []
    comparisons = (
        ("ALBERT vs. Annotator 1", "albert", "a1"), ("ALBERT vs. Annotator 2", "albert", "a2"),
        ("BioBERT vs. Annotator 1", "biobert", "a1"), ("BioBERT vs. Annotator 2", "biobert", "a2"),
        ("Protocol vs. Annotator 1", "protocol", "a1"), ("Protocol vs. Annotator 2", "protocol", "a2"),
        ("Annotator 1 vs. Annotator 2", "a1", "a2"),
    )

    def ratings(source: str, aspect: str) -> np.ndarray:
        if source in ("albert", "biobert"):
            return held_out[f"{source}_{aspect}_pred"].to_numpy(dtype=np.int8)
        return held_out[f"{source}_{aspect}"].to_numpy(dtype=np.int8)

    for name, left, right in comparisons:
        first = np.column_stack([ratings(left, a) for a in ASPECTS])
        second = np.column_stack([ratings(right, a) for a in ASPECTS])

        def macro_metrics(index: np.ndarray) -> np.ndarray:
            return np.nanmean(np.vstack([metrics(first[index, j], second[index, j]) for j in range(len(ASPECTS))]), axis=0)

        point = macro_metrics(np.arange(len(held_out)))
        ci = post_bootstrap(lambda idx: macro_metrics(idx)[2], len(held_out))
        rows.append({"comparison": name, "n_held_out_audit_posts": len(held_out),
                     "three_level_exact_agreement": float(point[0]), "linear_weighted_kappa": float(point[2]),
                     "weighted_kappa_ci_low": ci[0], "weighted_kappa_ci_high": ci[1],
                     "binary_exact_agreement": float(point[3]), "binary_kappa": float(point[4])})
    versus_annotators = pd.DataFrame(rows)
    return {"by_head": by_head, "means": means, "differences": differences, "versus_annotators": versus_annotators}


def reuse_agreement_with_protocol(prediction_csv: Path) -> pd.DataFrame:
    """Agreement of each model's predicted ratings with the protocol's ratings on the held-out test posts: kappa for
    presence, linear-weighted kappa for the three-level rating, and kappa for weak versus clear on the ratings both
    flagged. One row per dimension and a six-head mean, whose interval uses the same 1,000 shared test-post resamples
    (seed and count) as the six-head F1 comparison above."""
    predictions = pd.read_csv(prediction_csv, usecols=lambda column: column != "statement")
    truth = np.column_stack([predictions[f"{a}_true"].to_numpy(dtype=np.int8) for a in ASPECTS])
    models = {"ALBERT": "albert", "BioBERT": "biobert"}
    predicted = {label: np.column_stack([predictions[f"{prefix}_{a}_pred"].to_numpy(dtype=np.int8) for a in ASPECTS])
                 for label, prefix in models.items()}

    def by_head(label: str, index: np.ndarray) -> np.ndarray:
        values = []
        for j in range(len(ASPECTS)):
            target, model = truth[index, j], predicted[label][index, j]
            both = (target > 0) & (model > 0)
            values.append([cohen_kappa((target > 0).astype(np.int8), (model > 0).astype(np.int8), 2, linear=False),
                           cohen_kappa(target, model, 3, linear=True),
                           cohen_kappa(target[both] - 1, model[both] - 1, 2, linear=False) if both.any() else np.nan])
        return np.asarray(values)

    everyone = np.arange(len(truth))
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    draws = {label: [] for label in models}
    for _ in range(F1_BOOTSTRAP_REPLICATES):
        index = rng.integers(0, len(truth), len(truth))
        for label in models:
            draws[label].append(np.nanmean(by_head(label, index), axis=0))
    names = ("presence_kappa", "three_level_weighted_kappa", "weak_vs_clear_kappa")
    rows = []
    for label in models:
        heads = by_head(label, everyone)
        for j, aspect in enumerate(ASPECTS):
            rows.append({"rater": label, "dimension": aspect, "n_test_posts": len(truth), **dict(zip(names, heads[j]))})
        low, high = np.percentile(np.asarray(draws[label]), [2.5, 97.5], axis=0)
        mean_row = {"rater": label, "dimension": "mean of six heads", "n_test_posts": len(truth)}
        for k, name in enumerate(names):
            mean_row.update({name: float(np.nanmean(heads[:, k])), f"{name}_ci_low": float(low[k]), f"{name}_ci_high": float(high[k])})
        rows.append(mean_row)
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------
# Supplement: entropy and disagreement on the forced-choice label
# --------------------------------------------------------------------------

def single_label_entropy_tables(df: pd.DataFrame) -> dict[str, pd.DataFrame]:
    frame = df[["disagreement", "released_label", "protocol_label", "entropy_nats", "group_fold", "cluster"]].copy()
    frame["entropy_z"] = (frame["entropy_nats"] - frame["entropy_nats"].mean()) / frame["entropy_nats"].std(ddof=0)
    scores = {"Released category": np.empty(len(frame)), "Released category + entropy": np.empty(len(frame)),
              "Released category x entropy": np.empty(len(frame))}
    for fold in range(N_FOLDS):
        train = frame.loc[frame["group_fold"] != fold]
        held = frame.loc[frame["group_fold"] == fold]
        rates = train.groupby("released_label")["disagreement"].mean()
        scores["Released category"][held.index] = held["released_label"].map(rates).fillna(train["disagreement"].mean()).to_numpy()
        additive = smf.logit("disagreement ~ C(released_label) + entropy_z", data=train).fit(disp=0)
        scores["Released category + entropy"][held.index] = additive.predict(held).to_numpy()
        interaction = smf.logit("disagreement ~ C(released_label) * entropy_z", data=train).fit(disp=0)
        scores["Released category x entropy"][held.index] = interaction.predict(held).to_numpy()
    outcome = frame["disagreement"].to_numpy()
    cluster = frame["cluster"].to_numpy()
    n_clusters = int(cluster.max()) + 1
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    names = list(scores)
    draws = np.empty((AUC_BOOTSTRAP_REPLICATES, len(names) + 1))
    probabilities = np.full(n_clusters, 1.0 / n_clusters)
    for replicate in range(AUC_BOOTSTRAP_REPLICATES):
        weights = rng.multinomial(n_clusters, probabilities)[cluster]
        keep = weights > 0
        aucs = [roc_auc_score(outcome[keep], scores[name][keep], sample_weight=weights[keep]) for name in names]
        entropy_auc = roc_auc_score(outcome[keep], frame["entropy_nats"].to_numpy()[keep], sample_weight=weights[keep])
        draws[replicate] = [*aucs, entropy_auc]
    rows = [{"score": "Entropy alone", "evaluation": "no fitting", "auc": roc_auc_score(outcome, frame["entropy_nats"]),
             "auc_ci_low": np.percentile(draws[:, -1], 2.5), "auc_ci_high": np.percentile(draws[:, -1], 97.5)}]
    base = roc_auc_score(outcome, scores[names[0]])
    for j, name in enumerate(names):
        auc = roc_auc_score(outcome, scores[name])
        gain = draws[:, j] - draws[:, 0]
        rows.append({"score": name, "evaluation": "five grouped folds, out of fold", "auc": auc,
                     "auc_ci_low": np.percentile(draws[:, j], 2.5), "auc_ci_high": np.percentile(draws[:, j], 97.5),
                     "gain_over_category": auc - base,
                     "gain_ci_low": np.percentile(gain, 2.5) if j else np.nan, "gain_ci_high": np.percentile(gain, 97.5) if j else np.nan})
    auc_table = pd.DataFrame(rows)

    zero = frame.loc[frame["entropy_nats"] == 0]
    rows = [{"protocol_label": "ALL", "n_posts": len(zero), "share_of_corpus": len(zero) / len(frame),
             "disagreement_rate": float(zero["disagreement"].mean())}]
    for label in LABEL_ORDER:
        part = zero.loc[zero["protocol_label"] == label]
        rows.append({"protocol_label": label, "n_posts": len(part), "share_of_corpus": len(part) / len(frame),
                     "disagreement_rate": float(part["disagreement"].mean())})
    return {"auc": auc_table, "zero_block": pd.DataFrame(rows)}


# --------------------------------------------------------------------------
# Run, check and write
# --------------------------------------------------------------------------

# --------------------------------------------------------------------------
# Rule, continued: the label on posts with exactly two flagged dimensions, and the
# disagreements in which both labels name a category, against a base rate
# --------------------------------------------------------------------------

PAIR_ORDER = ("suicidal", "bipolar", "anxiety", "depression", "personality_disorder", "stress")  # Table 2, Panel B


def rule_pairwise_table(df: pd.DataFrame, flags: np.ndarray) -> pd.DataFrame:
    """For each pair of dimensions, posts with exactly those two flagged: the share labeled with each when the two
    ratings have the same strength, and the share labeled with the clearer one when one is clear and one weak."""
    cluster = df["cluster"].to_numpy()
    protocol = df["protocol_label"].to_numpy()
    k = df["n_dimensions"].to_numpy()
    levels = {aspect: df[f"level_{aspect}"].to_numpy() for aspect in ASPECTS}
    rows, numerators, denominators = [], [], []
    for i, first in enumerate(PAIR_ORDER):
        for second in PAIR_ORDER[i + 1:]:
            pair = (k == 2) & flags[:, ASPECTS.index(first)] & flags[:, ASPECTS.index(second)]
            rated = pair & (levels[first] > 0) & (levels[second] > 0)
            equal = rated & (levels[first] == levels[second])
            unequal = rated & (levels[first] != levels[second])
            clearer = np.where(levels[first] > levels[second], first.upper(), second.upper())
            rows.append({"dimension_a": first, "dimension_b": second, "n_posts": int(pair.sum()),
                         "n_equal_strength": int(equal.sum()), "n_unequal_strength": int(unequal.sum())})
            numerators += [equal & (protocol == first.upper()), equal & (protocol == second.upper()), unequal & (protocol == clearer)]
            denominators += [equal, equal, unequal]
    point, low, high = cluster_bootstrap_shares(cluster, np.column_stack(numerators), np.column_stack(denominators))
    table = pd.DataFrame(rows)
    for j, name in enumerate(("label_a_equal", "label_b_equal", "label_clearer_unequal")):
        table[f"share_{name}"], table[f"{name}_ci_low"], table[f"{name}_ci_high"] = point[j::3], low[j::3], high[j::3]
    return table


def rule_disagreement_pairs_table(df: pd.DataFrame, flags: np.ndarray, minimum: int = 200) -> pd.DataFrame:
    """Disagreements in which both labels name a category, by pair of labels: the share with the released category
    flagged, and the share expected from that category's flag rate on posts with the same AI label and another
    released label. Pairs with fewer than ``minimum`` disagreements are pooled."""
    cluster = df["cluster"].to_numpy()
    released = df["released_label"].to_numpy()
    protocol = df["protocol_label"].to_numpy()
    disagree = df["disagreement"].to_numpy().astype(bool)
    both_named = disagree & (released != "NORMAL") & (protocol != "NORMAL")
    names = [aspect.upper() for aspect in ASPECTS]
    pairs, columns = [], []
    for r in names:
        for a in names:
            if r == a:
                continue
            in_pair = both_named & (released == r) & (protocol == a)
            if not in_pair.any():
                continue
            flagged_r = flags[:, names.index(r)]
            base = (protocol == a) & (released != r)
            pairs.append((r, a, int(in_pair.sum())))
            columns += [in_pair, in_pair & flagged_r, base, base & flagged_r]
    columns = np.column_stack(columns)
    pairs_sorted = sorted(range(len(pairs)), key=lambda j: -pairs[j][2])
    shown = [j for j in pairs_sorted if pairs[j][2] >= minimum]
    pooled = [j for j in pairs_sorted if pairs[j][2] < minimum]

    def statistic(sums):
        n, f, bn, bf = sums[0::4], sums[1::4], sums[2::4], sums[3::4]
        base_rate = bf / bn
        values = []
        for group in ([[j] for j in shown] + [pooled, list(range(len(pairs)))]):
            values += [f[group].sum() / n[group].sum(), (n[group] * base_rate[group]).sum() / n[group].sum()]
        return np.array(values)

    point, low, high = cluster_bootstrap_statistic(cluster, columns, statistic)
    labels = [(pairs[j][0], pairs[j][1], pairs[j][2]) for j in shown]
    labels += [("OTHER", f"{len(pooled)} pairs", sum(pairs[j][2] for j in pooled)), ("ALL", "ALL", int(both_named.sum()))]
    return pd.DataFrame({
        "released_label": [x[0] for x in labels], "protocol_label": [x[1] for x in labels], "n_disagreements": [x[2] for x in labels],
        "share_released_flagged": point[0::2], "flagged_ci_low": low[0::2], "flagged_ci_high": high[0::2],
        "expected_share_from_base_rate": point[1::2], "expected_ci_low": low[1::2], "expected_ci_high": high[1::2],
    })


def audit_flag_rates(audit: pd.DataFrame) -> pd.DataFrame:
    """Each rater's share of the 300 audit posts on which each dimension is present (weak or clear)."""
    raters = (("protocol", "AI"), ("a1", "Annotator 1"), ("a2", "Annotator 2"))
    present = np.column_stack([(audit[f"{rater}_{aspect}"] > 0).to_numpy() for aspect in ASPECTS for rater, _n in raters]).astype(float)
    low, high = post_bootstrap(lambda index: present[index].mean(axis=0), len(audit))
    rows = [{"dimension": aspect, "rater": shown} for aspect in ASPECTS for _r, shown in raters]
    table = pd.DataFrame(rows)
    table["share_present"], table["ci_low"], table["ci_high"] = present.mean(axis=0), low, high
    return table


# --------------------------------------------------------------------------
# Sensitivity: only clear ratings counted; text length as a covariate
# --------------------------------------------------------------------------

def clear_only_sensitivity(df: pd.DataFrame, flags: np.ndarray) -> pd.DataFrame:
    """The main shares with a dimension counted when flagged (weak or clear) and when rated clear only."""
    cluster = df["cluster"].to_numpy()
    released = df["released_label"].to_numpy()
    protocol = df["protocol_label"].to_numpy()
    disagree = df["disagreement"].to_numpy().astype(bool)
    strata = df["entropy_stratum"].astype(str).to_numpy()
    stratum_names = [str(name) for name in df["entropy_stratum"].cat.categories]
    clear = np.column_stack([df[f"level_{aspect}"].to_numpy() == 2 for aspect in ASPECTS])
    definitions = (("any", flags), ("clear", clear))
    quantities = []  # (block, quantity, {definition: (numerator, denominator)})
    for name, present in definitions:
        k = present.sum(axis=1)
        released_present = np.zeros(len(df), dtype=bool)
        for i, aspect in enumerate(ASPECTS):
            mask = released == aspect.upper()
            released_present[mask] = present[mask, i]
        named = disagree & (released != "NORMAL") & (protocol != "NORMAL")
        items = [("co-occurrence", "two or more dimensions", k >= 2, np.ones(len(df), dtype=bool))]
        for aspect in PAIR_ORDER:
            eligible = (k >= 2) & present[:, ASPECTS.index(aspect)]
            items.append(("priority", aspect, eligible & (protocol == aspect.upper()), eligible))
        items += [
            ("disagreement type", "released label Normal", disagree & (released == "NORMAL"), disagree),
            ("disagreement type", "AI label Normal", disagree & (released != "NORMAL") & (protocol == "NORMAL"), disagree),
            ("disagreement type", "different category, released category also present", named & released_present, disagree),
            ("disagreement type", "different category, released category not present", named & ~released_present, disagree),
        ]
        items += [("entropy stratum", stratum, (strata == stratum) & (k >= 2), strata == stratum) for stratum in stratum_names]
        quantities.append((name, items))
    rows = []
    for (name, items) in quantities:
        point, low, high = cluster_bootstrap_shares(cluster, np.column_stack([it[2] for it in items]), np.column_stack([it[3] for it in items]))
        for j, (block, quantity, numerator, denominator) in enumerate(items):
            rows.append({"definition": name, "block": block, "quantity": quantity, "n_numerator": int(numerator.sum()),
                         "n_denominator": int(denominator.sum()), "share": float(point[j]), "ci_low": float(low[j]), "ci_high": float(high[j])})
    long = pd.DataFrame(rows)
    wide = long[long["definition"] == "any"].drop(columns="definition").merge(
        long[long["definition"] == "clear"].drop(columns="definition"), on=["block", "quantity"], suffixes=("_any", "_clear"))
    return wide


def length_adjustment(df: pd.DataFrame, audit: pd.DataFrame, cutpoints: pd.DataFrame) -> pd.DataFrame:
    """Logistic regressions of two or more dimensions, without and with the log word count of the post."""
    rows = []
    k = df["n_dimensions"].to_numpy()
    entropy = df["entropy_nats"].to_numpy()
    words = df["n_words"].to_numpy()
    rows += [
        {"sample": "corpus", "rater": "AI", "model": "descriptive", "term": "Spearman rho, word count and entropy",
         "value": float(stats.spearmanr(words, entropy)[0]), "n": len(df)},
        {"sample": "corpus", "rater": "AI", "model": "descriptive", "term": "Spearman rho, word count and number of dimensions",
         "value": float(stats.spearmanr(words, k)[0]), "n": len(df)},
    ]
    with_dimension = k >= 1
    log_words = np.log1p(words[with_dimension])
    frame = pd.DataFrame({
        "multi": (k[with_dimension] >= 2).astype(int),
        "entropy_z": (entropy[with_dimension] - entropy[with_dimension].mean()) / entropy[with_dimension].std(),
        "length_z": (log_words - log_words.mean()) / log_words.std(),
        "protocol_label": df.loc[with_dimension, "protocol_label"].to_numpy(),
    })
    groups = df["cluster"].to_numpy()[with_dimension]
    for model, formula in (("without length", "multi ~ entropy_z + C(protocol_label)"),
                           ("with length", "multi ~ entropy_z + length_z + C(protocol_label)")):
        fit = smf.logit(formula, data=frame).fit(disp=0, cov_type="cluster", cov_kwds={"groups": groups})
        ci = np.exp(fit.conf_int())
        for term, shown in (("entropy_z", "entropy (per SD)"), ("length_z", "log word count (per SD)")):
            if term in fit.params.index:
                rows.append({"sample": "corpus", "rater": "AI", "model": model, "term": shown, "value": float(np.exp(fit.params[term])),
                             "ci_low": float(ci.loc[term, 0]), "ci_high": float(ci.loc[term, 1]), "n": len(frame)})
    upper = float(cutpoints.loc[cutpoints["stratum"] == "positive_mid", "upper_nats"].iloc[0])
    audit_words = np.log1p(audit["n_words"].to_numpy())
    for rater, shown_rater in (("a1", "Annotator 1"), ("a2", "Annotator 2"), ("protocol", "AI")):
        audit_frame = pd.DataFrame({
            "multi": (audit[f"{rater}_n_dimensions"].to_numpy() >= 2).astype(int),
            "top": (audit["entropy_nats"].to_numpy() > upper).astype(int),
            "length_z": (audit_words - audit_words.mean()) / audit_words.std(),
        })
        for model, formula in (("without length", "multi ~ top"), ("with length", "multi ~ top + length_z")):
            fit = smf.logit(formula, data=audit_frame).fit(disp=0)
            ci = np.exp(fit.conf_int())
            for term, shown in (("top", "high entropy stratum"), ("length_z", "log word count (per SD)")):
                if term in fit.params.index:
                    rows.append({"sample": "audit", "rater": shown_rater, "model": model, "term": shown, "value": float(np.exp(fit.params[term])),
                                 "ci_low": float(ci.loc[term, 0]), "ci_high": float(ci.loc[term, 1]), "n": len(audit_frame)})
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------
# Four-LLM audit records: the label and the six evidence flags of every output
# --------------------------------------------------------------------------

MULTILLM_PROMPTS = ("minimal", "rubric", "cot")
PRIORITY_ORDER = ("suicidal", "bipolar", "anxiety", "depression", "personality_disorder", "stress")  # Table 2, Panel B


def read_multillm_records(path: Path, multillm_dir: Path) -> tuple[pd.DataFrame, float]:
    """Read the 21,600 crossed outputs; the label is the highest-scoring category. Returns the largest
    difference between the per-model flag rates here and those archived by the multi-LLM notebook."""
    flag_columns = [f"{aspect}_present" for aspect in ASPECTS]
    columns = ["item_id", "evaluator", "prompt_type", "temperature", "seed", "label", "text_hash", *flag_columns]
    records = pd.read_csv(path, usecols=columns, low_memory=False)
    if len(records) != 21_600 or records.duplicated(["item_id", "evaluator", "prompt_type", "temperature", "seed"]).any():
        raise ValueError("Expected 21,600 unique four-LLM records")
    for aspect, column in zip(ASPECTS, flag_columns):
        if records[column].dtype != bool:
            raise TypeError(f"{column} must be read as a boolean flag")
        records[aspect] = records[column].astype(int)
    records["n_flags"] = records[list(ASPECTS)].sum(axis=1)
    posts = sorted(records["item_id"].unique())
    records["post"] = records["item_id"].map({post: i for i, post in enumerate(posts)})
    archived = pd.read_csv(multillm_dir / "aspect_prevalence_by_model.csv").set_index(["model", "aspect"])["present_rate"]
    rates = records.groupby("evaluator")[list(ASPECTS)].mean().stack()
    return records, float(max(abs(rates.loc[key] - value) for key, value in archived.items()))


def multillm_sample_rows(df: pd.DataFrame, records: pd.DataFrame) -> pd.Series:
    """Corpus rows of the 100-post four-LLM audit sample, rebuilt as the multi-LLM notebook drew it (quartile strata of
    -sum(s * log2 s) on the raw scores, proportional allocation, random state 42) and confirmed by each statement's MD5."""
    raw = df.loc[:, list(PROB_COLS)].to_numpy(dtype=float)
    safe = np.where(raw > 0, raw, 1.0)
    bits = -(np.where(raw > 0, raw * np.log2(safe), 0.0)).sum(axis=1)
    q25, q50, q75 = pd.Series(bits).quantile([0.25, 0.50, 0.75]).to_numpy()
    stratum = pd.Series(np.select([bits <= q25, bits <= q50, bits <= q75], ["Q1", "Q2", "Q3"], "Q4"), index=df.index)
    counts = (stratum.value_counts(normalize=True).sort_index() * 100).round().astype(int)
    difference = 100 - int(counts.sum())
    if difference:
        counts.loc[counts.idxmax()] += difference
    rows = []
    for name, count in counts.items():
        rows += list(stratum.loc[stratum == name].to_frame().sample(n=int(count), random_state=42).index)
    items = [f"mh_{index:04d}" for index in range(len(rows))]
    expected = records.drop_duplicates("item_id").set_index("item_id").loc[items, "text_hash"].to_numpy()
    if not np.array_equal(df.loc[rows, "text_md5"].to_numpy(), expected):
        raise AssertionError("The rebuilt four-LLM sample does not match the audit records by statement hash")
    return pd.Series(rows, index=items)


def multillm_test_retest(df: pd.DataFrame, flags: np.ndarray, records: pd.DataFrame, rows: pd.Series) -> pd.DataFrame:
    """gpt-4o-mini under the protocol's own settings (rubric prompt, temperature 0, three seeds) against its original
    annotation of the same posts: exact agreement and kappa for the label and for each evidence flag."""
    rerun = records[(records["evaluator"] == "gpt-4o-mini") & (records["prompt_type"] == "rubric") & (records["temperature"] == 0)]
    corpus_rows = rows.loc[rerun["item_id"]].to_numpy()
    posts = rerun["post"].to_numpy()
    by_post = [np.flatnonzero(posts == post) for post in range(int(records["post"].max()) + 1)]
    pairs = [("label", label_codes(df.loc[corpus_rows, "protocol_label"]), label_codes(rerun["label"]), len(LABEL_ORDER))]
    pairs += [(aspect, flags[corpus_rows, ASPECTS.index(aspect)].astype(int), rerun[aspect].to_numpy().astype(int), 2) for aspect in ASPECTS]
    rows_out = []
    for name, original, again, levels in pairs:
        def statistic(index, original=original, again=again, levels=levels):
            take = np.concatenate([by_post[i] for i in index])
            return np.array([np.mean(original[take] == again[take]), cohen_kappa(original[take], again[take], levels, linear=False)])
        low, high = post_bootstrap(statistic, len(by_post))
        rows_out.append({"quantity": name, "n_outputs": len(rerun), "n_posts": len(by_post),
                         "exact_agreement": float(np.mean(original == again)), "exact_ci_low": float(low[0]), "exact_ci_high": float(high[0]),
                         "kappa": cohen_kappa(original, again, levels, linear=False), "kappa_ci_low": float(low[1]), "kappa_ci_high": float(high[1]),
                         "original_rate": float(np.mean(original)) if levels == 2 else np.nan,
                         "rerun_rate": float(np.mean(again)) if levels == 2 else np.nan})
    return pd.DataFrame(rows_out)


FOLLOW_UP_ROWS = np.arange(38_580, 38_600)   # the one batch that failed all attempts, re-annotated in two requests of 10


def multillm_rerun_request_order(df: pd.DataFrame, flags: np.ndarray, records: pd.DataFrame, rows: pd.Series) -> pd.DataFrame:
    """The original annotation sent 20 consecutive rows of the corpus file per request, and the file lists posts in blocks
    of one released label; the gpt-4o-mini rerun (rubric prompt, temperature 0, three seeds) sent the audit posts in
    requests of 20 audit posts. Agreement of each with the released label on the same 100 posts shows whether
    label-homogeneous requests raised that agreement; the share of posts with two or more flagged dimensions shows the same for
    co-occurring evidence. Neighbours are the other posts of a post's original request."""
    request = np.arange(len(df)) // 20
    request[FOLLOW_UP_ROWS] = request.max() + 1 + (FOLLOW_UP_ROWS - FOLLOW_UP_ROWS[0]) // 10
    released = df["released_label"].to_numpy()
    rerun = records[(records["evaluator"] == "gpt-4o-mini") & (records["prompt_type"] == "rubric") & (records["temperature"] == 0)]
    post = pd.DataFrame({"item_id": rows.index, "row": rows.to_numpy()})
    post["released"] = released[post["row"]]
    post["original_agrees"] = (df.loc[post["row"], "protocol_label"].to_numpy() == post["released"]).astype(float)
    post["original_two_or_more"] = ((flags[post["row"].to_numpy()] > 0).sum(axis=1) >= 2).astype(float)
    shares = []
    for row in post["row"]:
        mates = np.flatnonzero(request == request[row])
        mates = mates[mates != row]
        shares.append(float(np.mean(released[mates] == released[row])))
    post["share_neighbours_same_label"] = shares
    released_by_item = post.set_index("item_id")["released"]
    hits = (rerun["label"].to_numpy() == released_by_item.loc[rerun["item_id"]].to_numpy()).astype(float)
    post["rerun_agrees"] = post["item_id"].map(pd.Series(hits, index=rerun.index).groupby(rerun["item_id"].to_numpy()).mean())
    two = (rerun["n_flags"].to_numpy() >= 2).astype(float)
    post["rerun_two_or_more"] = post["item_id"].map(pd.Series(two, index=rerun.index).groupby(rerun["item_id"].to_numpy()).mean())
    if post["rerun_agrees"].isna().any() or rerun.groupby("item_id").size().ne(3).any():
        raise AssertionError("Each audit post must have three rerun outputs")
    groups = {
        "all audit posts": np.ones(len(post), dtype=bool),
        "neighbours at least 75% of the same released label": post["share_neighbours_same_label"].to_numpy() >= 0.75,
        "neighbours below 75% of the same released label": post["share_neighbours_same_label"].to_numpy() < 0.75,
    }
    out = []
    for name, mask in groups.items():
        part = post.loc[mask].reset_index(drop=True)
        difference = (part["rerun_agrees"] - part["original_agrees"]).to_numpy()
        low, high = post_bootstrap(lambda index: difference[index].mean(), len(part))
        difference_two = (part["rerun_two_or_more"] - part["original_two_or_more"]).to_numpy()
        low_two, high_two = post_bootstrap(lambda index: difference_two[index].mean(), len(part))
        out.append({"posts": name, "n_posts": len(part),
                     "mean_share_neighbours_same_label": float(part["share_neighbours_same_label"].mean()),
                     "share_posts_all_neighbours_same_label": float((part["share_neighbours_same_label"] == 1).mean()),
                     "original_agreement_with_released": float(part["original_agrees"].mean()),
                     "rerun_agreement_with_released": float(part["rerun_agrees"].mean()),
                     "difference_rerun_minus_original": float(difference.mean()), "difference_ci_low": low, "difference_ci_high": high,
                     "original_two_or_more": float(part["original_two_or_more"].mean()), "rerun_two_or_more": float(part["rerun_two_or_more"].mean()),
                     "difference_two_or_more": float(difference_two.mean()), "difference_two_or_more_ci_low": low_two,
                     "difference_two_or_more_ci_high": high_two})
    return pd.DataFrame(out)


def multillm_priority_tables(records: pd.DataFrame) -> dict[str, pd.DataFrame]:
    """Among outputs with two or more flagged dimensions that include a dimension, the share whose label is that
    dimension, under each prompt; and the paired difference from the minimal prompt (same post resamples)."""
    n_posts = int(records["post"].max()) + 1
    shares, differences = [], []
    for aspect in PRIORITY_ORDER:
        hits, totals = {}, {}
        for prompt in MULTILLM_PROMPTS:
            sub = records[(records["prompt_type"] == prompt) & (records["n_flags"] >= 2) & (records[aspect] == 1)]
            hits[prompt] = np.bincount(sub["post"], weights=(sub["label"] == aspect.upper()).astype(float), minlength=n_posts)
            totals[prompt] = np.bincount(sub["post"], minlength=n_posts).astype(float)

        def share_vector(index, hits=hits, totals=totals):
            weights = np.bincount(index, minlength=n_posts)
            return np.array([float((weights * hits[p]).sum() / (weights * totals[p]).sum()) for p in MULTILLM_PROMPTS])

        point = np.array([hits[p].sum() / totals[p].sum() for p in MULTILLM_PROMPTS])
        low, high = post_bootstrap(share_vector, n_posts)
        d_low, d_high = post_bootstrap(lambda index: share_vector(index)[1:] - share_vector(index)[0], n_posts)
        for i, prompt in enumerate(MULTILLM_PROMPTS):
            shares.append({"dimension": aspect, "prompt": prompt, "n_outputs": int(totals[prompt].sum()),
                           "share": float(point[i]), "share_ci_low": float(low[i]), "share_ci_high": float(high[i])})
        for i, prompt in enumerate(MULTILLM_PROMPTS[1:]):
            differences.append({"dimension": aspect, "prompt": prompt, "difference_from_minimal": float(point[i + 1] - point[0]),
                                "ci_low": float(d_low[i]), "ci_high": float(d_high[i])})
    for prompt in MULTILLM_PROMPTS:
        shares.append({"dimension": "two or more flagged", "prompt": prompt,
                       "n_outputs": int(((records["prompt_type"] == prompt) & (records["n_flags"] >= 2)).sum()),
                       "share": np.nan, "share_ci_low": np.nan, "share_ci_high": np.nan})
    return {"by_prompt": pd.DataFrame(shares), "differences": pd.DataFrame(differences)}


# --------------------------------------------------------------------------
# Supplement: logistic regressions of disagreement, model-based and cluster-robust
# --------------------------------------------------------------------------

LOGIT_FORMULAS = {
    "entropy_only": "disagreement ~ entropy_std",
    "entropy_plus_released_status": "disagreement ~ entropy_std + C(released_label, Treatment(reference='NORMAL'))",
}


def logistic_cluster_robust(df: pd.DataFrame) -> dict[str, pd.DataFrame]:
    """Refit the two regressions of disagreement with model-based and cleaned-statement cluster-robust errors."""
    frame = df[["disagreement", "released_label", "entropy_nats", "cluster"]].copy()
    frame["entropy_std"] = (frame["entropy_nats"] - frame["entropy_nats"].mean()) / frame["entropy_nats"].std(ddof=1)
    coefficients, models = [], []
    for model, formula in LOGIT_FORMULAS.items():
        usual = smf.logit(formula, data=frame).fit(disp=0)
        robust = smf.logit(formula, data=frame).fit(disp=0, cov_type="cluster", cov_kwds={"groups": frame["cluster"].to_numpy()})
        usual_ci, robust_ci = np.exp(usual.conf_int()), np.exp(robust.conf_int())
        for term in usual.params.index:
            coefficients.append({
                "model": model, "term": term,
                "coefficient_log_odds": float(usual.params[term]), "odds_ratio": float(np.exp(usual.params[term])),
                "model_based_se": float(usual.bse[term]), "model_based_p_value": float(usual.pvalues[term]),
                "model_based_or_ci_95_lower": float(usual_ci.loc[term, 0]), "model_based_or_ci_95_upper": float(usual_ci.loc[term, 1]),
                "cluster_robust_se": float(robust.bse[term]), "cluster_robust_p_value": float(robust.pvalues[term]),
                "cluster_robust_or_ci_95_lower": float(robust_ci.loc[term, 0]), "cluster_robust_or_ci_95_upper": float(robust_ci.loc[term, 1]),
            })
        models.append({"model": model, "formula": formula, "n_rows": int(len(frame)),
                       "n_cleaned_statement_clusters": int(frame["cluster"].nunique()),
                       "maximum_cluster_size": int(frame["cluster"].value_counts().max()),
                       "covariance": "cleaned-statement cluster-robust sandwich", "small_sample_correction": True})
    return {"coefficients": pd.DataFrame(coefficients), "models": pd.DataFrame(models)}


# --------------------------------------------------------------------------
# Trust, continued: the four LLMs by scope, agreement on strength where both raters flagged a
# dimension, and the differences between annotator-protocol and annotator-annotator agreement
# --------------------------------------------------------------------------

RAW_ASPECT_NAMES = tuple(aspect.upper() for aspect in ASPECTS)
MULTILLM_SAMPLE_SIZE = 100
MULTILLM_SCOPES = ("all 54 conditions", "rubric prompt, temperature 0")


def multillm_raw_aspects(path: Path) -> pd.DataFrame:
    """One row per four-LLM record and dimension, with the strength read from raw_json under the
    multi-LLM notebook's rules: a strength other than none, weak, or clear is invalid, and an entry
    is schema-consistent when its strength is valid and agrees with its presence flag."""
    keys = ["item_id", "prompt_type", "temperature", "seed", "evaluator"]
    flag_columns = [f"{aspect}_present" for aspect in ASPECTS]
    frame = pd.read_csv(path, usecols=[*keys, "raw_json", *flag_columns], low_memory=False)
    rows: list[dict[str, object]] = []
    for record in frame.itertuples(index=False):
        aspects = json.loads(record.raw_json).get("aspects")
        if not isinstance(aspects, dict) or set(aspects) != set(RAW_ASPECT_NAMES):
            raise ValueError(f"Invalid aspect schema for {record.item_id}")
        base = {key: getattr(record, key) for key in keys}
        for aspect, name in zip(ASPECTS, RAW_ASPECT_NAMES):
            entry = aspects[name]
            if not isinstance(entry, dict) or not {"present", "strength", "cues"}.issubset(entry):
                raise ValueError(f"Invalid {name} object for {record.item_id}")
            present = entry["present"]
            if not isinstance(present, bool) or present != bool(getattr(record, f"{aspect}_present")):
                raise ValueError(f"Raw and flattened {name} presence disagree for {record.item_id}")
            strength = "" if entry["strength"] is None else str(entry["strength"]).strip().lower()
            code = LEVEL_TO_INT.get(strength, -1)
            rows.append({**base, "aspect": aspect, "strength_code": code, "present": present,
                         "consistent": bool(code >= 0 and present == (code > 0))})
    return pd.DataFrame(rows)


def contingency_metrics(counts: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Return exact, nominal-kappa, and linear-kappa arrays from count tables (as in the multi-LLM notebook)."""
    counts = np.asarray(counts, dtype=float)
    n_levels = counts.shape[-1]
    totals = counts.sum(axis=(-2, -1))
    diagonal = np.trace(counts, axis1=-2, axis2=-1)
    with np.errstate(divide="ignore", invalid="ignore"):
        exact = diagonal / totals
        row_margins = counts.sum(axis=-1)
        column_margins = counts.sum(axis=-2)
        expected_agreement = (row_margins * column_margins).sum(axis=-1) / totals**2
        nominal = (exact - expected_agreement) / (1.0 - expected_agreement)

        disagreement_weights = (
            np.abs(np.arange(n_levels)[:, None] - np.arange(n_levels)[None, :])
            / max(n_levels - 1, 1)
        )
        observed_disagreement = (
            counts * disagreement_weights
        ).sum(axis=(-2, -1)) / totals
        expected_tables = (
            row_margins[..., :, None] * column_margins[..., None, :] / totals[..., None, None]
        )
        expected_disagreement = (
            expected_tables * disagreement_weights
        ).sum(axis=(-2, -1)) / totals
        linear = 1.0 - observed_disagreement / expected_disagreement

    exact = np.where(totals > 0, exact, np.nan)
    nominal = np.where((totals > 0) & (np.abs(1.0 - expected_agreement) > 1e-15), nominal, np.nan)
    linear = np.where((totals > 0) & (expected_disagreement > 1e-15), linear, np.nan)
    return exact, nominal, linear


def multillm_agreement_by_scope(raw: pd.DataFrame, subsets: dict[str, pd.DataFrame] | None = None) -> pd.DataFrame:
    """Means over the 36 model-pair-by-dimension cells of matched outputs, for all 54 conditions and for the
    protocol's own prompt and temperature: three-level rating (schema-consistent entries; linear-weighted
    kappa), presence (nominal kappa), and weak versus clear where both models flagged the dimension
    (schema-consistent entries; nominal kappa). Intervals resample the 100 posts with the multi-LLM
    notebook's bootstrap draws, so the all-condition values reproduce its archived intervals. Other subsets of the
    records (named scopes) can be passed in ``subsets``; the posts and the resamples stay the same."""
    items = sorted(raw["item_id"].unique())
    if len(items) != MULTILLM_SAMPLE_SIZE:
        raise ValueError("Expected 100 four-LLM audit posts")
    lookup = {item: index for index, item in enumerate(items)}
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    draws = rng.integers(0, MULTILLM_SAMPLE_SIZE, size=(BOOTSTRAP_REPLICATES, MULTILLM_SAMPLE_SIZE))
    weights = np.stack([np.bincount(row, minlength=MULTILLM_SAMPLE_SIZE) for row in draws]).astype(np.int16)
    models = sorted(raw["evaluator"].unique())
    keys = ["item_id", "prompt_type", "temperature", "seed", "aspect"]
    if subsets is None:
        subsets = {
            MULTILLM_SCOPES[0]: raw,
            MULTILLM_SCOPES[1]: raw.loc[(raw["prompt_type"] == "rubric") & (raw["temperature"] == 0)],
        }

    def counts(item: np.ndarray, first: np.ndarray, second: np.ndarray, mask: np.ndarray, levels: int) -> np.ndarray:
        table = np.zeros((MULTILLM_SAMPLE_SIZE, levels, levels), dtype=np.int32)
        np.add.at(table, (item[mask], first[mask], second[mask]), 1)
        return table

    rows: list[dict[str, object]] = []
    for scope, data in subsets.items():
        cells: dict[str, list[np.ndarray]] = {"three_level": [], "presence": [], "strength_both_flagged": []}
        for model_a, model_b in itertools.combinations(models, 2):
            paired = data.loc[data["evaluator"] == model_a].merge(
                data.loc[data["evaluator"] == model_b], on=keys, suffixes=("_a", "_b"), validate="one_to_one")
            for aspect in ASPECTS:
                cell = paired.loc[paired["aspect"] == aspect]
                item = cell["item_id"].map(lookup).to_numpy(dtype=int)
                first = cell["strength_code_a"].to_numpy(dtype=int)
                second = cell["strength_code_b"].to_numpy(dtype=int)
                consistent = cell["consistent_a"].to_numpy(dtype=bool) & cell["consistent_b"].to_numpy(dtype=bool)
                present_a = cell["present_a"].to_numpy(dtype=bool)
                present_b = cell["present_b"].to_numpy(dtype=bool)
                cells["three_level"].append(counts(item, first, second, consistent, 3))
                cells["presence"].append(counts(item, present_a.astype(int), present_b.astype(int), np.ones(len(cell), dtype=bool), 2))
                cells["strength_both_flagged"].append(counts(item, first - 1, second - 1, consistent & present_a & present_b, 2))
        row: dict[str, object] = {"scope": scope, "n_pair_dimension_cells": len(cells["presence"]),
                                  "n_matched_outputs_per_cell": int(cells["presence"][0].sum())}
        for name, kappa_position in (("three_level", 2), ("presence", 1), ("strength_both_flagged", 1)):
            stacked = np.stack(cells[name])
            point = contingency_metrics(stacked.sum(axis=1))
            boot = contingency_metrics(np.einsum("bi,cijk->bcjk", weights, stacked, optimize=True))
            row[f"{name}_n_ratings"] = int(stacked.sum())
            for label, position in (("exact", 0), ("kappa", kappa_position)):
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", RuntimeWarning)
                    replicate_means = np.nanmean(boot[position], axis=1)
                finite = replicate_means[np.isfinite(replicate_means)]
                low, high = np.percentile(finite, [2.5, 97.5])
                row[f"{name}_{label}"] = float(np.nanmean(point[position]))
                row[f"{name}_{label}_ci_low"] = float(low)
                row[f"{name}_{label}_ci_high"] = float(high)
                row[f"{name}_{label}_valid_replicates"] = int(len(finite))
        rows.append(row)
    return pd.DataFrame(rows)


MULTILLM_FACETS = (
    ("prompt", "prompt_type", ("minimal", "rubric", "cot")),
    ("temperature", "temperature", (0.0, 0.2, 0.4, 0.6, 0.8, 1.0)),
    ("seed", "seed", (1001, 2001, 3001)),
)


def multillm_agreement_by_facet(raw: pd.DataFrame) -> pd.DataFrame:
    """The agreement among the four LLMs on the aspect ratings within each prompt, temperature, and seed, each level
    pooled over the other two facets, with the statistics and resamples of ``multillm_agreement_by_scope``."""
    subsets, levels = {}, []
    for facet, column, values in MULTILLM_FACETS:
        for value in values:
            subsets[f"{facet} {value}"] = raw.loc[raw[column] == value]
            levels.append((facet, str(value)))
    table = multillm_agreement_by_scope(raw, subsets)
    table.insert(1, "facet", [facet for facet, _value in levels])
    table.insert(2, "level", [value for _facet, value in levels])
    return table


THREE_LEVELS = ("none", "weak", "clear")


def specific_agreement(counts: np.ndarray) -> np.ndarray:
    """Agreement on each level from two-rater count tables of shape (..., k, k): 2 n_jj / (n_j+ + n_+j), the share of
    the ratings at level j, given by either rater, that the other rater matched. NaN where neither rater used level j."""
    counts = np.asarray(counts, dtype=float)
    diagonal = np.diagonal(counts, axis1=-2, axis2=-1)
    margins = counts.sum(axis=-1) + counts.sum(axis=-2)
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(margins > 0, 2.0 * diagonal / margins, np.nan)


def chance_agreement(counts: np.ndarray) -> np.ndarray:
    """The agreement on each level expected if the two raters assigned levels independently with their observed
    frequencies: 2 n_j+ n_+j / (N (n_j+ + n_+j)). NaN where neither rater used level j."""
    counts = np.asarray(counts, dtype=float)
    rows, columns = counts.sum(axis=-1), counts.sum(axis=-2)
    total = counts.sum(axis=(-2, -1))[..., None]
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(rows + columns > 0, 2.0 * rows * columns / (total * (rows + columns)), np.nan)


def multillm_level_agreement(raw: pd.DataFrame) -> pd.DataFrame:
    """Agreement among the four LLMs on each level of the three-level rating, from the same schema-consistent matched
    outputs, cells, scopes, and bootstrap draws as ``multillm_agreement_by_scope``: per model-pair-by-dimension cell,
    then the mean over the 36 cells (a cell in which neither model used a level is left out of that level's mean).
    Rows for each dimension give the mean over the six model pairs, all conditions, without intervals. ``chance`` is
    the mean of the cells' agreement expected under independence. Each cell's level agreements, weighted by the
    level's share of ratings, must reproduce its exact agreement."""
    items = sorted(raw["item_id"].unique())
    if len(items) != MULTILLM_SAMPLE_SIZE:
        raise ValueError("Expected 100 four-LLM audit posts")
    lookup = {item: index for index, item in enumerate(items)}
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    draws = rng.integers(0, MULTILLM_SAMPLE_SIZE, size=(BOOTSTRAP_REPLICATES, MULTILLM_SAMPLE_SIZE))
    weights = np.stack([np.bincount(row, minlength=MULTILLM_SAMPLE_SIZE) for row in draws]).astype(np.int16)
    models = sorted(raw["evaluator"].unique())
    keys = ["item_id", "prompt_type", "temperature", "seed", "aspect"]
    subsets = {
        MULTILLM_SCOPES[0]: raw,
        MULTILLM_SCOPES[1]: raw.loc[(raw["prompt_type"] == "rubric") & (raw["temperature"] == 0)],
    }
    rows: list[dict[str, object]] = []
    for scope, data in subsets.items():
        cells, cell_aspects = [], []
        for model_a, model_b in itertools.combinations(models, 2):
            paired = data.loc[data["evaluator"] == model_a].merge(
                data.loc[data["evaluator"] == model_b], on=keys, suffixes=("_a", "_b"), validate="one_to_one")
            for aspect in ASPECTS:
                cell = paired.loc[paired["aspect"] == aspect]
                consistent = cell["consistent_a"].to_numpy(dtype=bool) & cell["consistent_b"].to_numpy(dtype=bool)
                item = cell["item_id"].map(lookup).to_numpy(dtype=int)[consistent]
                first = cell["strength_code_a"].to_numpy(dtype=int)[consistent]
                second = cell["strength_code_b"].to_numpy(dtype=int)[consistent]
                table = np.zeros((MULTILLM_SAMPLE_SIZE, 3, 3), dtype=np.int32)
                np.add.at(table, (item, first, second), 1)
                cells.append(table)
                cell_aspects.append(aspect)
        stacked = np.stack(cells)
        pooled = stacked.sum(axis=1).astype(float)
        point = specific_agreement(pooled)
        chance = chance_agreement(pooled)
        share = (pooled.sum(axis=-1) + pooled.sum(axis=-2)) / (2.0 * pooled.sum(axis=(-2, -1)))[:, None]
        if not np.allclose(np.nansum(point * share, axis=1), np.trace(pooled, axis1=-2, axis2=-1) / pooled.sum(axis=(-2, -1)), atol=1e-12):
            raise AssertionError(f"{scope}: level agreements do not reproduce the exact agreement")
        boot = specific_agreement(np.einsum("bi,cijk->bcjk", weights, stacked, optimize=True))
        for j, level in enumerate(THREE_LEVELS):
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", RuntimeWarning)
                replicate_means = np.nanmean(boot[..., j], axis=1)
            finite = replicate_means[np.isfinite(replicate_means)]
            low, high = np.percentile(finite, [2.5, 97.5])
            rows.append({"scope": scope, "dimension": "mean of 36 cells", "level": level,
                         "n_cells": len(cells), "n_cells_with_level": int(np.isfinite(point[:, j]).sum()),
                         "n_ratings": int(pooled.sum()), "agreement": float(np.nanmean(point[:, j])),
                         "ci_low": float(low), "ci_high": float(high), "valid_replicates": int(len(finite)),
                         "chance": float(np.nanmean(chance[:, j]))})
        if scope == MULTILLM_SCOPES[0]:
            aspects = np.asarray(cell_aspects)
            for aspect in ASPECTS:
                part, part_chance = point[aspects == aspect], chance[aspects == aspect]
                for j, level in enumerate(THREE_LEVELS):
                    with warnings.catch_warnings():
                        warnings.simplefilter("ignore", RuntimeWarning)
                        value = float(np.nanmean(part[:, j]))
                        expected = float(np.nanmean(part_chance[:, j]))
                    rows.append({"scope": scope, "dimension": aspect, "level": level, "n_cells": int(len(part)),
                                 "n_cells_with_level": int(np.isfinite(part[:, j]).sum()),
                                 "n_ratings": int(pooled[aspects == aspect].sum()), "agreement": value,
                                 "ci_low": np.nan, "ci_high": np.nan, "valid_replicates": np.nan, "chance": expected})
    return pd.DataFrame(rows)


AUDIT_COMPARISONS = (
    ("Annotator 1 vs. Annotator 2", "a1", "a2"),
    ("Annotator 1 vs. protocol", "a1", "protocol"),
    ("Annotator 2 vs. protocol", "a2", "protocol"),
)


def audit_strength_and_differences(audit: pd.DataFrame) -> dict[str, pd.DataFrame]:
    """Audit sample: weak versus clear where both raters coded a dimension present (exact agreement and
    nominal kappa per dimension, then the mean over the six), and the annotator-protocol minus
    annotator-annotator difference in the six-dimension mean of linear-weighted and presence kappa.
    Both use the same 2,000 post resamples, so the differences are paired."""
    levels = {rater: {aspect: audit[f"{rater}_{aspect}"].to_numpy(dtype=int) for aspect in ASPECTS}
              for rater in ("a1", "a2", "protocol")}

    def strength(index: np.ndarray) -> np.ndarray:
        values = []
        for _name, left, right in AUDIT_COMPARISONS:
            exact, kappa = [], []
            for aspect in ASPECTS:
                first, second = levels[left][aspect][index], levels[right][aspect][index]
                both = (first > 0) & (second > 0)
                if not both.any():
                    exact.append(np.nan)
                    kappa.append(np.nan)
                    continue
                exact.append(float(np.mean(first[both] == second[both])))
                kappa.append(cohen_kappa(first[both] - 1, second[both] - 1, 2, linear=False))
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", RuntimeWarning)
                values += [np.nanmean(exact), np.nanmean(kappa)]
        return np.asarray(values, dtype=float)

    linear_position = METRIC_NAMES.index("three_level_linear_weighted_kappa")
    binary_position = METRIC_NAMES.index("binary_nominal_kappa")

    def means(index: np.ndarray) -> dict[str, np.ndarray]:
        return {name: np.nanmean(np.vstack([metrics(levels[left][a][index], levels[right][a][index]) for a in ASPECTS]), axis=0)
                for name, left, right in AUDIT_COMPARISONS}

    def differences(index: np.ndarray) -> np.ndarray:
        values = means(index)
        reference = values[AUDIT_COMPARISONS[0][0]]
        return np.asarray([values[name][position] - reference[position]
                           for position in (linear_position, binary_position)
                           for name, _left, _right in AUDIT_COMPARISONS[1:]], dtype=float)

    everyone = np.arange(len(audit))
    point = strength(everyone)
    low, high = post_bootstrap(strength, len(audit))
    rows = []
    for k, (name, left, right) in enumerate(AUDIT_COMPARISONS):
        n_both = int(sum(((levels[left][a] > 0) & (levels[right][a] > 0)).sum() for a in ASPECTS))
        rows.append({"comparison": name, "n_ratings_both_present": n_both,
                     "exact_agreement": point[2 * k], "exact_ci_low": low[2 * k], "exact_ci_high": high[2 * k],
                     "kappa": point[2 * k + 1], "kappa_ci_low": low[2 * k + 1], "kappa_ci_high": high[2 * k + 1]})
    strength_table = pd.DataFrame(rows)

    base = means(everyone)
    point = differences(everyone)
    low, high = post_bootstrap(differences, len(audit))
    rows = []
    k = 0
    for metric, position in (("linear-weighted kappa, three-level rating", linear_position), ("kappa, presence", binary_position)):
        for name, _left, _right in AUDIT_COMPARISONS[1:]:
            rows.append({"metric": metric, "comparison": name, "reference": AUDIT_COMPARISONS[0][0],
                         "kappa_comparison": float(base[name][position]), "kappa_reference": float(base[AUDIT_COMPARISONS[0][0]][position]),
                         "difference": point[k], "difference_ci_low": low[k], "difference_ci_high": high[k]})
            k += 1
    return {"strength": strength_table, "differences": pd.DataFrame(rows)}


def audit_level_agreement(audit: pd.DataFrame) -> pd.DataFrame:
    """Audit sample: agreement on each level of the three-level rating for each comparison, per dimension and as the
    mean over the six dimensions, with 2,000 post resamples for the means; ``chance`` is the agreement expected under
    independence. Each dimension's level agreements, weighted by the level's share of ratings, must reproduce its
    three-level exact agreement."""
    levels = {rater: {aspect: audit[f"{rater}_{aspect}"].to_numpy(dtype=int) for aspect in ASPECTS}
              for rater in ("a1", "a2", "protocol")}

    def tables(index: np.ndarray, left: str, right: str) -> np.ndarray:
        out = np.zeros((len(ASPECTS), 3, 3))
        for k, aspect in enumerate(ASPECTS):
            np.add.at(out[k], (levels[left][aspect][index], levels[right][aspect][index]), 1.0)
        return out

    def statistic(index: np.ndarray) -> np.ndarray:
        values = []
        for _name, left, right in AUDIT_COMPARISONS:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", RuntimeWarning)
                values += list(np.nanmean(specific_agreement(tables(index, left, right)), axis=0))
        return np.asarray(values, dtype=float)

    everyone = np.arange(len(audit))
    point = statistic(everyone)
    low, high = post_bootstrap(statistic, len(audit))
    rows = []
    for c, (name, left, right) in enumerate(AUDIT_COMPARISONS):
        per_dimension = tables(everyone, left, right)
        agreement = specific_agreement(per_dimension)
        chance = chance_agreement(per_dimension)
        share = (per_dimension.sum(axis=-1) + per_dimension.sum(axis=-2)) / (2.0 * per_dimension.sum(axis=(-2, -1)))[:, None]
        exact = np.asarray([metrics(levels[left][a], levels[right][a])[0] for a in ASPECTS])
        if not np.allclose(np.nansum(agreement * share, axis=1), exact, atol=1e-12):
            raise AssertionError(f"{name}: level agreements do not reproduce the exact agreement")
        for j, level in enumerate(THREE_LEVELS):
            rows.append({"comparison": name, "dimension": "mean of six", "level": level,
                         "agreement": point[3 * c + j], "ci_low": low[3 * c + j], "ci_high": high[3 * c + j],
                         "chance": float(np.nanmean(chance[:, j]))})
        for k, aspect in enumerate(ASPECTS):
            for j, level in enumerate(THREE_LEVELS):
                rows.append({"comparison": name, "dimension": aspect, "level": level,
                             "agreement": float(agreement[k, j]), "ci_low": np.nan, "ci_high": np.nan, "chance": float(chance[k, j])})
    return pd.DataFrame(rows)



def weak_partners(counts: np.ndarray) -> np.ndarray:
    """For two-rater three-level count tables (..., 3, 3): of the WEAK ratings given by either rater, the shares on which
    the other rater gave NONE, WEAK, and CLEAR. The three shares sum to one; NaN where neither rater used WEAK."""
    counts = np.asarray(counts, dtype=float)
    total = counts[..., 1, :].sum(axis=-1) + counts[..., :, 1].sum(axis=-1)
    parts = np.stack([counts[..., 1, 0] + counts[..., 0, 1], 2.0 * counts[..., 1, 1], counts[..., 1, 2] + counts[..., 2, 1]], axis=-1)
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(total[..., None] > 0, parts / total[..., None], np.nan)


def weak_rating_partners(raw: pd.DataFrame, audit: pd.DataFrame) -> pd.DataFrame:
    """Where a WEAK rating went: the four LLMs (schema-consistent matched outputs, per model-pair-by-dimension cell, mean
    of the 36 cells, for all conditions and the protocol's settings) and the audit comparisons (per dimension, mean of
    the six). The WEAK share equals the agreement on that level (``trust_multillm_level_agreement.csv`` and
    ``trust_level_agreement_audit.csv``)."""
    rows = []
    models = sorted(raw["evaluator"].unique())
    keys = ["item_id", "prompt_type", "temperature", "seed", "aspect"]
    scopes = {MULTILLM_SCOPES[0]: raw, MULTILLM_SCOPES[1]: raw.loc[(raw["prompt_type"] == "rubric") & (raw["temperature"] == 0)]}
    for scope, data in scopes.items():
        tables = []
        for model_a, model_b in itertools.combinations(models, 2):
            paired = data.loc[data["evaluator"] == model_a].merge(
                data.loc[data["evaluator"] == model_b], on=keys, suffixes=("_a", "_b"), validate="one_to_one")
            for aspect in ASPECTS:
                cell = paired.loc[paired["aspect"] == aspect]
                consistent = cell["consistent_a"].to_numpy(dtype=bool) & cell["consistent_b"].to_numpy(dtype=bool)
                table = np.zeros((3, 3))
                np.add.at(table, (cell["strength_code_a"].to_numpy(dtype=int)[consistent], cell["strength_code_b"].to_numpy(dtype=int)[consistent]), 1.0)
                tables.append(table)
        shares = weak_partners(np.stack(tables))
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            mean = np.nanmean(shares, axis=0)
        rows.append({"source": "four LLMs", "comparison": scope, "n_units": len(tables), "n_units_with_weak": int(np.isfinite(shares[:, 0]).sum()),
                     "share_none": float(mean[0]), "share_weak": float(mean[1]), "share_clear": float(mean[2])})
    levels = {rater: {aspect: audit[f"{rater}_{aspect}"].to_numpy(dtype=int) for aspect in ASPECTS} for rater in ("a1", "a2", "protocol")}
    for name, left, right in AUDIT_COMPARISONS:
        tables = np.zeros((len(ASPECTS), 3, 3))
        for k, aspect in enumerate(ASPECTS):
            np.add.at(tables[k], (levels[left][aspect], levels[right][aspect]), 1.0)
        shares = weak_partners(tables)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            mean = np.nanmean(shares, axis=0)
        rows.append({"source": "audit", "comparison": name, "n_units": len(ASPECTS), "n_units_with_weak": int(np.isfinite(shares[:, 0]).sum()),
                     "share_none": float(mean[0]), "share_weak": float(mean[1]), "share_clear": float(mean[2])})
    out = pd.DataFrame(rows)
    if not np.allclose(out[["share_none", "share_weak", "share_clear"]].sum(axis=1), 1.0, atol=1e-12):
        raise AssertionError("The shares of where a weak rating went do not sum to one")
    return out


def blank_statement_outputs(df: pd.DataFrame, flags: np.ndarray) -> pd.DataFrame:
    """What the protocol returned for posts whose text field is blank."""
    blank = df["blank_statement"].to_numpy(dtype=bool)
    n_flags = (flags[blank] > 0).sum(axis=1)
    released, protocol = label_codes(df["released_label"]), label_codes(df["protocol_label"])
    agree = released == protocol
    two = (flags > 0).sum(axis=1) >= 2
    return pd.DataFrame([{
        "n_blank_posts": int(blank.sum()),
        "n_released_label_not_normal": int((df.loc[blank, "released_label"] != "NORMAL").sum()),
        "n_protocol_label_normal": int((df.loc[blank, "protocol_label"] == "NORMAL").sum()),
        "n_with_a_flagged_dimension": int((n_flags > 0).sum()),
        "n_with_two_or_more_flagged_dimensions": int((n_flags >= 2).sum()),
        "n_with_positive_entropy": int((df.loc[blank, "entropy_nats"] > 0).sum()),
        **{f"{name}_{scope}": value for scope, keep in (("all_posts", np.ones(len(df), dtype=bool)), ("without_blank_posts", ~blank))
           for name, value in (("label_agreement", float(agree[keep].mean())), ("label_kappa", cohen_kappa(released[keep], protocol[keep], len(LABEL_ORDER), linear=False)),
                               ("share_two_or_more", float(two[keep].mean())), ("n_disagreements", int((~agree[keep]).sum())))},
    }])


def acceptance_checks_trust_extension(multillm_scope: pd.DataFrame, audit_extra: dict[str, pd.DataFrame],
                                      trust: dict[str, pd.DataFrame], blanks: pd.DataFrame, multillm_dir: Path) -> list[dict[str, object]]:
    """The new quantities must reproduce what the archived tables already report for the same data."""
    checks: list[dict[str, object]] = []

    def check(name: str, observed, expected) -> None:
        ok = observed == expected
        checks.append({"check": name, "observed": observed, "expected": expected, "passed": bool(ok)})
        if not ok:
            raise AssertionError(f"Acceptance check failed: {name}: observed {observed}, expected {expected}")

    archived = pd.read_csv(multillm_dir / "aspect_level_agreement_robustness.csv").query("scope == 'overall_macro_pair_aspect'")
    three = archived.query("representation == 'strength_3level' and inclusion_definition == 'schema_consistent'").iloc[0]
    presence = archived.query("representation == 'binary_presence'").iloc[0]
    both = pd.read_csv(multillm_dir / "aspect_strength_both_present_agreement.csv").query("scope == 'overall_macro_pair_aspect'").iloc[0]
    ours = multillm_scope.loc[multillm_scope["scope"] == MULTILLM_SCOPES[0]].iloc[0]
    pairs = [
        (ours["three_level_exact"], three["exact_agreement"]), (ours["three_level_exact_ci_low"], three["exact_agreement_ci_low"]),
        (ours["three_level_exact_ci_high"], three["exact_agreement_ci_high"]), (ours["three_level_kappa"], three["linear_weighted_kappa"]),
        (ours["three_level_kappa_ci_low"], three["linear_weighted_kappa_ci_low"]), (ours["three_level_kappa_ci_high"], three["linear_weighted_kappa_ci_high"]),
        (ours["presence_exact"], presence["exact_agreement"]), (ours["presence_kappa"], presence["nominal_kappa"]),
        (ours["presence_kappa_ci_low"], presence["nominal_kappa_ci_low"]), (ours["presence_kappa_ci_high"], presence["nominal_kappa_ci_high"]),
        (ours["strength_both_flagged_exact"], both["exact_agreement"]), (ours["strength_both_flagged_exact_ci_low"], both["exact_agreement_ci_low"]),
        (ours["strength_both_flagged_exact_ci_high"], both["exact_agreement_ci_high"]),
    ]
    check("four-LLM agreement over all 54 conditions reproduces the archived values and intervals",
          bool(max(abs(float(a) - float(b)) for a, b in pairs) < 1e-12), True)
    check("weak-versus-clear ratings compared over all conditions", int(ours["strength_both_flagged_n_ratings"]), int(both["n_both_present_schema_consistent"]))
    per_dimension = trust["per_dimension"].set_index("comparison")["three_level_linear_weighted_kappa"]
    differences = audit_extra["differences"].query("metric == 'linear-weighted kappa, three-level rating'")
    reference = float(per_dimension["Annotator 1 vs. Annotator 2"])
    observed = [round(float(row["kappa_comparison"]) - reference, 12) for _, row in differences.iterrows()]
    expected = [round(float(per_dimension[name]) - reference, 12) for name in ("Annotator 1 vs. AI aspects", "Annotator 2 vs. AI aspects")]
    check("audit kappa differences start from the archived per-dimension kappas", observed, expected)
    check("blank posts", int(blanks["n_blank_posts"].iloc[0]), 362)
    return checks


def acceptance_checks(df, audit, qc, counts, pairs, reuse, trust, logistic, prevalence_difference,
                      pairwise, disagreement_pairs, rule, signal, clear_only, length, test_retest) -> list[dict[str, object]]:
    """Values already reported in the manuscript must be reproduced before new ones are trusted."""
    checks: list[dict[str, object]] = []

    def check(name: str, observed, expected) -> None:
        ok = observed == expected
        checks.append({"check": name, "observed": observed, "expected": expected, "passed": bool(ok)})
        if not ok:
            raise AssertionError(f"Acceptance check failed: {name}: observed {observed}, expected {expected}")

    corpus = qc.loc[qc["sample"] == "full corpus"].iloc[0]
    check("records", int(len(df)), EXPECTED_N)
    check("cleaned-statement clusters", int(df["cluster"].nunique()), 51_330)
    entropy_only = logistic["coefficients"].query("model == 'entropy_only' and term == 'entropy_std'").iloc[0]
    check("entropy-only odds ratio per SD (3 dp)", round(float(entropy_only["odds_ratio"]), 3), 1.311)
    check("four-LLM records reproduce the archived per-model flag rates", bool(prevalence_difference < 1e-12), True)
    check("posts with exactly two flagged dimensions, summed over the 15 pairs",
          int(pairwise["n_posts"].sum()), int(counts.loc[counts["n_dimensions"] == "2", "n_posts"].iloc[0]))
    total = disagreement_pairs.loc[disagreement_pairs["released_label"] == "ALL"].iloc[0]
    check("two-category disagreements with the released category flagged",
          (int(total["n_disagreements"]), int(round(total["share_released_flagged"] * total["n_disagreements"]))), (12_124, 5_506))
    reported = np.concatenate([
        counts.loc[counts["n_dimensions"] == "2 or more", "share"].to_numpy(),
        rule["priority"].set_index("dimension").loc[list(PAIR_ORDER), "share_forced_choice_is_dimension"].to_numpy(),
        rule["types"]["share_of_disagreements"].to_numpy()[[0, 1, 2, 3]],
        signal["by_stratum"]["share_two_or_more_dimensions"].to_numpy()[:-1],
    ])
    check("weak-or-clear column of the clear-only table reproduces the reported shares",
          bool(np.max(np.abs(clear_only["share_any"].to_numpy() - reported)) < 1e-12), True)
    unadjusted = length.query("sample == 'corpus' and model == 'without length' and term == 'entropy (per SD)'")["value"].iloc[0]
    check("entropy odds ratio without length equals the reported value (3 dp)", round(float(unadjusted), 3), 2.31)
    check("test-retest outputs: 100 posts x 3 seeds", int(test_retest["n_outputs"].iloc[0]), 300)
    check("released-protocol agreements", int(corpus["n_agree"]), 33_267)
    check("released-protocol kappa (3 dp)", round(float(corpus["kappa"]), 3), 0.517)
    check("zero-entropy posts", int((df["entropy_nats"] == 0).sum()), 27_890)
    check("share with two or more dimensions (3 dp)", round(float(counts.loc[counts["n_dimensions"] == "2 or more", "share"].iloc[0]), 3), 0.333)
    top_pair = pairs.iloc[0]
    check("largest pair", (top_pair["dimension_a"], top_pair["dimension_b"], int(top_pair["n_posts_both"])), ("depression", "suicidal", 9_610))
    human = qc.loc[qc["sample"] == "human audit (enriched)"].set_index("comparison")["n_agree"].astype(int).to_dict()
    check("audit primary-label agreements", [human[name] for name in (
        "Annotator 1 vs. Annotator 2", "Annotator 1 vs. protocol", "Annotator 2 vs. protocol",
        "Annotator 1 vs. released label", "Annotator 2 vs. released label", "Protocol vs. released label")],
        [169, 155, 164, 158, 145, 138])
    heads = reuse["by_head"]
    albert = heads.loc[heads["rater"] == "ALBERT"].set_index("dimension")["weighted_f1"]
    check("ALBERT weighted F1 by head (3 dp)", [round(float(albert[a]), 3) for a in ASPECTS], [0.809, 0.853, 0.882, 0.703, 0.974, 0.956])
    means = reuse["means"].set_index("rater")
    check("mean weighted F1, ALBERT and BioBERT (3 dp)", [round(float(means.loc[m, "weighted_f1"]), 3) for m in ("ALBERT", "BioBERT")], [0.863, 0.857])
    check("mean macro F1, ALBERT and BioBERT (3 dp)", [round(float(means.loc[m, "macro_f1"]), 3) for m in ("ALBERT", "BioBERT")], [0.593, 0.589])
    llm = qc.loc[qc["check"] == "other LLMs"]
    check("fixed-condition LLM agreement range (2 dp)", [round(float(llm["agreement"].min()), 2), round(float(llm["agreement"].max()), 2)], [0.72, 0.80])
    check("annotator per-dimension agreement matches the archived table", int(len(trust["per_dimension"])), 3)
    return checks


def run_analysis(
    labels_csv: Path,
    audit_dir: Path,
    prediction_csv: Path,
    human_agreement_csv: Path,
    multillm_dir: Path,
    output_dir: Path,
    project_root: Path,
    multillm_records_csv: Path,
) -> dict[str, object]:
    """Run every block, write the aggregate tables, and return the manifest."""
    output_dir.mkdir(parents=True, exist_ok=True)
    df, flags, cutpoints = load_corpus(labels_csv)
    audit = read_audit(audit_dir)
    records, prevalence_difference = read_multillm_records(multillm_records_csv, multillm_dir)

    qc = quality_control_table(df, audit, multillm_dir, records)
    counts, pairs = profile_tables(df, flags)
    rule = rule_tables(df, flags)
    pairwise = rule_pairwise_table(df, flags)
    disagreement_pairs = rule_disagreement_pairs_table(df, flags)
    signal = signal_tables(df, audit, cutpoints)
    trust = trust_tables(audit, human_agreement_csv, multillm_dir)
    flag_rates = audit_flag_rates(audit)
    clear_only = clear_only_sensitivity(df, flags)
    length = length_adjustment(df, audit, cutpoints)
    reuse = reuse_tables(prediction_csv, audit)
    reuse_agreement = reuse_agreement_with_protocol(prediction_csv)
    single = single_label_entropy_tables(df)
    logistic = logistic_cluster_robust(df)
    priority = multillm_priority_tables(records)
    sample_rows = multillm_sample_rows(df, records)
    test_retest = multillm_test_retest(df, flags, records, sample_rows)
    request_order = multillm_rerun_request_order(df, flags, records, sample_rows)
    multillm_raw = multillm_raw_aspects(multillm_records_csv)
    multillm_scope = multillm_agreement_by_scope(multillm_raw)
    multillm_facets = multillm_agreement_by_facet(multillm_raw)
    multillm_levels = multillm_level_agreement(multillm_raw)
    audit_levels = audit_level_agreement(audit)
    weak_partner_shares = weak_rating_partners(multillm_raw, audit)
    for source, levels_table, key_column, mean_label in (("four LLMs", multillm_levels, "scope", "mean of 36 cells"),
                                                         ("audit", audit_levels, "comparison", "mean of six")):
        for _, r in weak_partner_shares[weak_partner_shares["source"] == source].iterrows():
            weak = levels_table[(levels_table[key_column] == r["comparison"]) & (levels_table["dimension"] == mean_label) & (levels_table["level"] == "weak")]["agreement"].iloc[0]
            if abs(weak - r["share_weak"]) > 1e-12:
                raise AssertionError(f"{r['comparison']}: the matched share of weak ratings differs from the agreement on that level")
    audit_extra = audit_strength_and_differences(audit)
    blanks = blank_statement_outputs(df, flags)
    checks = acceptance_checks(df, audit, qc, counts, pairs, reuse, trust, logistic, prevalence_difference,
                               pairwise, disagreement_pairs, rule, signal, clear_only, length, test_retest)
    checks += acceptance_checks_trust_extension(multillm_scope, audit_extra, trust, blanks, multillm_dir)
    per_cell = {facet: sorted({int(v) for v in group["n_matched_outputs_per_cell"]}) for facet, group in multillm_facets.groupby("facet")}
    if len(multillm_facets) != 12 or per_cell != {"prompt": [1800], "seed": [1800], "temperature": [900]}:
        raise AssertionError(f"Unexpected four-LLM facet subsets: {per_cell}")
    checks.append({"check": "four-LLM agreement by facet: 12 levels; matched outputs per cell 1,800 (prompt, seed) and 900 (temperature)",
                   "observed": str(per_cell), "expected": "{'prompt': [1800], 'seed': [1800], 'temperature': [900]}", "passed": True})
    checks.append({"check": "agreement on each level reproduces the exact agreement of every four-LLM cell and audit dimension",
                   "observed": True, "expected": True, "passed": True})
    original = float(request_order.loc[request_order['posts'] == 'all audit posts', 'original_agreement_with_released'].iloc[0])
    if round(1 - original, 3) != 0.350:
        raise AssertionError(f'Released-AI disagreement on the 100 audit posts is {1 - original:.3f}, not the reported 0.350')
    checks.append({'check': 'released-AI disagreement on the 100-post audit sample (3 dp)', 'observed': round(1 - original, 3), 'expected': 0.35, 'passed': True})

    tables = {
        "qc_single_label_agreement.csv": qc,
        "profile_dimension_counts.csv": counts,
        "profile_pair_counts.csv": pairs,
        "rule_priority_shares.csv": rule["priority"],
        "rule_disagreement_by_category_and_dimensions.csv": rule["by_category"],
        "rule_depression_suicidal_contrast.csv": rule["contrast"],
        "rule_disagreement_types.csv": rule["types"],
        "rule_pairwise_two_flagged_dimensions.csv": pairwise,
        "rule_disagreement_pairs_against_base_rate.csv": disagreement_pairs,
        "multillm_priority_by_prompt.csv": priority["by_prompt"],
        "multillm_priority_difference_from_minimal.csv": priority["differences"],
        "multillm_test_retest.csv": test_retest,
        "multillm_rerun_request_order.csv": request_order,
        "signal_cooccurrence_by_entropy_stratum.csv": signal["by_stratum"],
        "signal_entropy_by_dimension_count.csv": signal["by_count"],
        "signal_annotator_coded_by_stratum.csv": signal["annotators"],
        "signal_supplementary_indices.csv": signal["extra"],
        "signal_entropy_stratum_cutpoints.csv": cutpoints,
        "trust_per_dimension_agreement.csv": trust["per_dimension"],
        "trust_two_or_more_dimensions_agreement.csv": trust["two_or_more"],
        "trust_overlap_in_disagreements.csv": trust["overlap"],
        "trust_multillm_summary.csv": trust["multillm"],
        "trust_flag_rates_by_rater.csv": flag_rates,
        "trust_multillm_agreement_by_scope.csv": multillm_scope,
        "trust_multillm_agreement_by_facet.csv": multillm_facets,
        "trust_strength_where_both_present.csv": audit_extra["strength"],
        "trust_differences_from_annotator_agreement.csv": audit_extra["differences"],
        "trust_multillm_level_agreement.csv": multillm_levels,
        "trust_level_agreement_audit.csv": audit_levels,
        "trust_weak_rating_partners.csv": weak_partner_shares,
        "protocol_blank_statements.csv": blanks,
        "sensitivity_clear_only.csv": clear_only,
        "sensitivity_length_adjustment.csv": length,
        "reuse_by_head.csv": reuse["by_head"],
        "reuse_six_head_means.csv": reuse["means"],
        "reuse_difference_from_always_none.csv": reuse["differences"],
        "reuse_versus_annotators_held_out.csv": reuse["versus_annotators"],
        "reuse_agreement_with_protocol.csv": reuse_agreement,
        "supplement_disagreement_auc_out_of_fold.csv": single["auc"],
        "supplement_zero_entropy_block.csv": single["zero_block"],
        "supplement_logistic_coefficients.csv": logistic["coefficients"],
        "supplement_logistic_models.csv": logistic["models"],
    }
    for name, table in tables.items():
        _write_csv(table, output_dir / name)

    def relative(path: Path) -> str:
        try:
            return str(path.resolve().relative_to(project_root.resolve()))
        except ValueError:
            return path.name

    inputs = [
        labels_csv,
        audit_dir / "generated_subset" / "human_validation_sample_key_300.csv",
        audit_dir / "outputs" / "human_validation_annotator1_300.xlsm",
        audit_dir / "outputs" / "human_validation_annotator2_300.xlsm",
        prediction_csv,
        human_agreement_csv,
        multillm_dir / "hard_label_pairwise_robustness.csv",
        multillm_dir / "aspect_level_agreement_robustness.csv",
        multillm_dir / "aspect_count_pairwise_robustness.csv",
        multillm_dir / "cooccurrence_pattern_pairwise_robustness.csv",
        multillm_dir / "aspect_prevalence_by_model.csv",
        multillm_dir / "aspect_strength_both_present_agreement.csv",
        multillm_records_csv,
    ]
    input_hashes = pd.DataFrame([{"file": relative(path), "bytes": path.stat().st_size, "sha256": sha256_file(path)} for path in inputs])
    _write_csv(input_hashes, output_dir / "input_sha256.csv")
    output_hashes = _hash_outputs(output_dir, tables.keys())
    _write_csv(output_hashes, output_dir / "output_sha256.csv")

    types = rule["types"].set_index("disagreement_type")["share_of_disagreements"]
    strata = signal["by_stratum"].set_index("entropy_stratum")["share_two_or_more_dimensions"]
    manifest = {
        "analysis_id": "mental_health_multivariate_evidence",
        "analysis_script": NOTEBOOK_NAME,
        "analysis_script_sha256": "n/a (code inlined in notebook)",
        "models_trained": 0,
        "provider_calls": 0,
        "cluster_definition": "cleaned statement (lowercase; URL/user/hashtag token normalization; punctuation removal; whitespace collapse); each post with a blank cleaned statement is its own cluster",
        "bootstrap": {
            "cluster_and_post_resamples": BOOTSTRAP_REPLICATES,
            "f1_shared_post_resamples": F1_BOOTSTRAP_REPLICATES,
            "auc_cluster_resamples": AUC_BOOTSTRAP_REPLICATES,
            "seed": BOOTSTRAP_SEED,
        },
        "n_records": int(len(df)),
        "n_cleaned_statement_clusters": int(df["cluster"].nunique()),
        "n_disagreements": int(df["disagreement"].sum()),
        "n_audit_posts": int(len(audit)),
        "n_held_out_audit_posts": int(reuse["versus_annotators"]["n_held_out_audit_posts"].iloc[0]),
        "headline": {
            "share_two_or_more_dimensions": float(counts.loc[counts["n_dimensions"] == "2 or more", "share"].iloc[0]),
            "share_of_disagreements_from_cooccurring_evidence": float(types.filter(like="co-occurring").iloc[0]),
            "share_two_or_more_at_zero_entropy": float(strata["H=0"]),
            "share_two_or_more_in_top_entropy_stratum": float(strata["positive_high"]),
        },
        "acceptance_assertions": {"status": "passed", "n_checks": len(checks), "checks": checks},
        "input_hashes": "input_sha256.csv",
        "output_hashes": "output_sha256.csv",
        "output_file_count_excluding_manifest_and_hash_index": len(tables),
        "software": {
            "python": platform.python_version(),
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "scipy": scipy.__version__,
            "scikit_learn": sklearn.__version__,
            "statsmodels": statsmodels.__version__,
        },
    }
    (output_dir / "multivariate_evidence_manifest.json").write_text(json.dumps(manifest, indent=2, sort_keys=True, default=str) + "\n")
    return manifest


In [3]:
manifest = run_analysis(
    LABELS_CSV, HUMAN_AUDIT_DIR, ASPECT_PREDICTIONS_CSV, HUMAN_AGREEMENT_CSV, MULTILLM_OUTPUT, OUTPUT, MENTAL_HEALTH_ROOT,
    multillm_records_csv=MULTILLM_RECORDS_CSV,
)
print(json.dumps({key: value for key, value in manifest.items() if key != 'acceptance_assertions'}, indent=2, sort_keys=True))
print(f"Acceptance checks: {manifest['acceptance_assertions']['status']} ({manifest['acceptance_assertions']['n_checks']} checks)")

{
  "analysis_id": "mental_health_multivariate_evidence",
  "analysis_script": "MentalHealth_multivariate_evidence.ipynb",
  "analysis_script_sha256": "n/a (code inlined in notebook)",
  "bootstrap": {
    "auc_cluster_resamples": 1000,
    "cluster_and_post_resamples": 2000,
    "f1_shared_post_resamples": 1000,
    "seed": 2025
  },
  "cluster_definition": "cleaned statement (lowercase; URL/user/hashtag token normalization; punctuation removal; whitespace collapse); each post with a blank cleaned statement is its own cluster",
  "headline": {
    "share_of_disagreements_from_cooccurring_evidence": 0.27841828478964403,
    "share_two_or_more_at_zero_entropy": 0.23327357475797778,
    "share_two_or_more_dimensions": 0.333446449107328,
    "share_two_or_more_in_top_entropy_stratum": 0.6132989817200344
  },
  "input_hashes": "input_sha256.csv",
  "models_trained": 0,
  "n_audit_posts": 300,
  "n_cleaned_statement_clusters": 51330,
  "n_disagreements": 19776,
  "n_held_out_audit_posts": 6

In [4]:
# 1. Quality control of the forced-choice label
table = pd.read_csv(OUTPUT / "qc_single_label_agreement.csv")
display(table.drop(columns=["interval"]).round(4))
display(pd.read_csv(OUTPUT / "multillm_test_retest.csv").round(4))
display(pd.read_csv(OUTPUT / "protocol_blank_statements.csv"))
display(pd.read_csv(OUTPUT / "multillm_rerun_request_order.csv").round(4))

,check,comparison,sample,n,n_agree,agreement,kappa,kappa_ci_low,kappa_ci_high
0,released labels,Protocol vs. released label,full corpus,53043,33267,0.6272,0.5167,0.5113,0.5219
1,human coders,Annotator 1 vs. Annotator 2,human audit (enriched),300,169,0.5633,0.4893,0.4242,0.5517
2,human coders,Annotator 1 vs. protocol,human audit (enriched),300,155,0.5167,0.4362,0.3673,0.5025
3,human coders,Annotator 2 vs. protocol,human audit (enriched),300,164,0.5467,0.4624,0.3924,0.5280
4,human coders,Annotator 1 vs. released label,human audit (enriched),300,158,0.5267,0.4493,0.3851,0.5134
5,human coders,Annotator 2 vs. released label,human audit (enriched),300,145,0.4833,0.4010,0.3393,0.4628
6,human coders,Protocol vs. released label,human audit (enriched),300,138,0.4600,0.3730,0.3087,0.4390
7,other LLMs,gemini-2.5-flash vs. gpt-4o-mini,"multi-LLM audit, rubric prompt, temperature 0, seed 1001",100,80,0.8000,0.7385,0.6265,0.8400
8,other LLMs,gemini-2.5-flash vs. llama-70b,"multi-LLM audit, rubric prompt, temperature 0, seed 1001",100,75,0.7500,0.6831,0.5734,0.7816
9,other LLMs,gemini-2.5-flash vs. mistral-small,"multi-LLM audit, rubric prompt, temperature 0, seed 1001",100,75,0.7500,0.6788,0.5690,0.7798


,quantity,n_outputs,n_posts,exact_agreement,exact_ci_low,exact_ci_high,kappa,kappa_ci_low,kappa_ci_high,original_rate,rerun_rate
0,label,300,100,0.8733,0.8033,0.9333,0.8351,0.7430,0.9107,NaN,NaN
1,depression,300,100,0.9000,0.8400,0.9567,0.7886,0.6583,0.9032,0.41,0.3500
2,anxiety,300,100,0.9000,0.8433,0.9501,0.7178,0.5474,0.8610,0.24,0.2200
3,suicidal,300,100,0.9467,0.9000,0.9833,0.8614,0.7402,0.9569,0.26,0.2600
4,stress,300,100,0.7900,0.7133,0.8567,0.4081,0.1920,0.5958,0.17,0.2800
5,bipolar,300,100,0.9800,0.9500,1.0000,0.7899,0.2827,1.0000,0.04,0.0600
6,personality_disorder,300,100,0.9833,0.9567,1.0000,0.6973,-0.0101,1.0000,0.03,0.0267


,n_blank_posts,n_released_label_not_normal,n_protocol_label_normal,n_with_a_flagged_dimension,n_with_two_or_more_flagged_dimensions,n_with_positive_entropy,label_agreement_all_posts,label_kappa_all_posts,share_two_or_more_all_posts,n_disagreements_all_posts,label_agreement_without_blank_posts,label_kappa_without_blank_posts,share_two_or_more_without_blank_posts,n_disagreements_without_blank_posts
0,362,354,360,3,1,5,0.62717,0.516651,0.333446,19776,0.631309,0.521573,0.335719,19423


,posts,n_posts,mean_share_neighbours_same_label,share_posts_all_neighbours_same_label,original_agreement_with_released,rerun_agreement_with_released,difference_rerun_minus_original,difference_ci_low,difference_ci_high,original_two_or_more,rerun_two_or_more,difference_two_or_more,difference_two_or_more_ci_low,difference_two_or_more_ci_high
0,all audit posts,100,0.7805,0.5500,0.6500,0.6733,0.0233,-0.0367,0.0801,0.3400,0.3633,0.0233,-0.0400,0.0900
1,neighbours at least 75% of the same released label,57,0.9935,0.9649,0.6316,0.6842,0.0526,-0.0526,0.1579,0.2105,0.2047,-0.0058,-0.0703,0.0585
2,neighbours below 75% of the same released label,43,0.4982,0.0000,0.6744,0.6589,-0.0155,-0.0469,0.0000,0.5116,0.5736,0.0620,-0.0543,0.1860


In [5]:
# 2. Profile: number of flagged dimensions, and the most frequent pairs
display(pd.read_csv(OUTPUT / "profile_dimension_counts.csv").round(4))
display(pd.read_csv(OUTPUT / "profile_pair_counts.csv").round(4))

,n_dimensions,n_posts,share,share_ci_low,share_ci_high
0,0,17463,0.3292,0.3250,0.3335
1,1,17893,0.3373,0.3333,0.3415
2,2,13514,0.2548,0.2511,0.2586
3,3,3384,0.0638,0.0617,0.0659
4,4 or more,789,0.0149,0.0138,0.0159
5,2 or more,17687,0.3334,0.3292,0.3376


,dimension_a,dimension_b,n_posts_both,share_of_corpus
0,depression,suicidal,9610,0.1812
1,depression,anxiety,4424,0.0834
2,depression,stress,4367,0.0823
3,anxiety,stress,3218,0.0607
4,anxiety,suicidal,1907,0.0360
5,suicidal,stress,1291,0.0243
6,depression,bipolar,1022,0.0193
7,depression,personality_disorder,657,0.0124
8,anxiety,bipolar,520,0.0098
9,anxiety,personality_disorder,357,0.0067


In [6]:
# 3. Rule: priority among co-occurring dimensions (corpus, then the four-LLM audit by prompt)
display(pd.read_csv(OUTPUT / "rule_priority_shares.csv").round(4))
display(pd.read_csv(OUTPUT / "rule_depression_suicidal_contrast.csv").round(4))
display(pd.read_csv(OUTPUT / "multillm_priority_by_prompt.csv").round(4))
display(pd.read_csv(OUTPUT / "multillm_priority_difference_from_minimal.csv").round(4))
display(pd.read_csv(OUTPUT / "rule_pairwise_two_flagged_dimensions.csv").round(4))


,dimension,n_posts_with_dimension_among_two_or_more,share_forced_choice_is_dimension,share_ci_low,share_ci_high
0,suicidal,9992,0.8924,0.8863,0.8982
1,bipolar,1310,0.5435,0.5156,0.5721
2,anxiety,6659,0.4616,0.4492,0.4735
3,depression,15163,0.2829,0.2757,0.2904
4,personality_disorder,879,0.1650,0.1405,0.1905
5,stress,6406,0.0695,0.0628,0.0761


,evidence,released_label,n_posts,disagreement_rate,rate_ci_low,rate_ci_high,share_forced_choice_suicidal,share_ci_low,share_ci_high
0,depression and suicidal both flagged,SUICIDAL,5218,0.0569,0.0509,0.0630,0.9431,0.9370,0.9491
1,depression and suicidal both flagged,DEPRESSION,3842,0.8639,0.8527,0.8750,0.8503,0.8389,0.8615
2,depression and suicidal both flagged,ALL,9610,0.4270,0.4171,0.4366,0.8946,0.8884,0.9007


,dimension,prompt,n_outputs,share,share_ci_low,share_ci_high
0,suicidal,minimal,1596,0.6460,0.5211,0.7583
1,suicidal,rubric,1595,0.8150,0.7232,0.8814
2,suicidal,cot,1448,0.9199,0.8776,0.9516
3,bipolar,minimal,382,0.6021,0.3026,0.7972
4,bipolar,rubric,368,0.5272,0.2177,0.7643
5,bipolar,cot,282,0.5426,0.1497,0.9122
6,anxiety,minimal,1841,0.3248,0.2130,0.4364
7,anxiety,rubric,1693,0.2894,0.1737,0.4142
8,anxiety,cot,2327,0.2656,0.1681,0.3649
9,depression,minimal,2653,0.4440,0.3443,0.5450


,dimension,prompt,difference_from_minimal,ci_low,ci_high
0,suicidal,rubric,0.1691,0.1042,0.2319
1,suicidal,cot,0.2739,0.1827,0.3662
2,bipolar,rubric,-0.0749,-0.1696,0.0140
3,bipolar,cot,-0.0595,-0.3003,0.2264
4,anxiety,rubric,-0.0354,-0.0809,0.0111
5,anxiety,cot,-0.0592,-0.1058,-0.0140
6,depression,rubric,-0.0839,-0.1367,-0.0311
7,depression,cot,-0.1252,-0.1831,-0.0653
8,personality_disorder,rubric,-0.1005,-0.1489,-0.0334
9,personality_disorder,cot,-0.0403,-0.1219,0.0260


,dimension_a,dimension_b,n_posts,n_equal_strength,n_unequal_strength,share_label_a_equal,label_a_equal_ci_low,label_a_equal_ci_high,share_label_b_equal,label_b_equal_ci_low,label_b_equal_ci_high,share_label_clearer_unequal,label_clearer_unequal_ci_low,label_clearer_unequal_ci_high
0,suicidal,bipolar,17,11,6,0.9091,0.6923,1.0000,0.0909,0.0000,0.3077,1.0000,1.0000,1.0000
1,suicidal,anxiety,185,112,73,0.9643,0.9262,0.9918,0.0357,0.0082,0.0738,0.9726,0.9296,1.0000
2,suicidal,depression,6918,5882,1036,0.9810,0.9771,0.9843,0.0190,0.0157,0.0229,0.9855,0.9779,0.9921
3,suicidal,personality_disorder,25,7,18,1.0000,1.0000,1.0000,0.0000,0.0000,0.0000,1.0000,1.0000,1.0000
4,suicidal,stress,108,52,56,1.0000,1.0000,1.0000,0.0000,0.0000,0.0000,0.9643,0.9107,1.0000
5,bipolar,anxiety,128,54,74,0.6852,0.5577,0.8085,0.2778,0.1613,0.4001,0.9730,0.9286,1.0000
6,bipolar,depression,370,180,190,0.8778,0.8272,0.9259,0.1111,0.0656,0.1594,0.9895,0.9725,1.0000
7,bipolar,personality_disorder,23,11,12,0.6364,0.3000,0.9167,0.2727,0.0000,0.6000,1.0000,1.0000,1.0000
8,bipolar,stress,66,23,43,0.4783,0.2778,0.6842,0.1304,0.0000,0.2859,0.9767,0.9250,1.0000
9,anxiety,depression,1535,449,1085,0.3831,0.3384,0.4307,0.5991,0.5513,0.6452,0.9991,0.9965,1.0000


In [7]:
# 3. Rule: disagreement by released category and number of flagged dimensions (rate, then n)
by_category = pd.read_csv(OUTPUT / "rule_disagreement_by_category_and_dimensions.csv")
order = list(dict.fromkeys(by_category["released_label"]))
display(by_category.pivot(index="released_label", columns="n_dimensions", values="disagreement_rate").loc[order].round(4))
display(by_category.pivot(index="released_label", columns="n_dimensions", values="n_posts").loc[order])
display(by_category.round(4))

n_dimensions,0,1,2 or more
released_label,,,
NORMAL,0.0001,0.6213,0.9714
DEPRESSION,1.0000,0.4343,0.6149
SUICIDAL,1.0000,0.3812,0.1534
ANXIETY,0.9970,0.0583,0.0953
STRESS,1.0000,0.6338,0.8358
BIPOLAR,1.0000,0.5348,0.5321
PERSONALITY_DISORDER,1.0000,0.8427,0.9579
ALL,0.2449,0.4330,0.4382


n_dimensions,0,1,2 or more
released_label,,,
NORMAL,13186,2955,210
DEPRESSION,1545,6222,7637
SUICIDAL,811,3804,6038
ANXIETY,331,2298,1259
STRESS,591,1256,822
BIPOLAR,610,1021,1246
PERSONALITY_DISORDER,389,337,475
ALL,17463,17893,17687


,released_label,n_dimensions,n_posts,disagreement_rate,rate_ci_low,rate_ci_high
0,NORMAL,0,13186,0.0001,0.0000,0.0002
1,NORMAL,1,2955,0.6213,0.6032,0.6403
2,NORMAL,2 or more,210,0.9714,0.9474,0.9911
3,DEPRESSION,0,1545,1.0000,1.0000,1.0000
4,DEPRESSION,1,6222,0.4343,0.4230,0.4471
5,DEPRESSION,2 or more,7637,0.6149,0.6038,0.6258
6,SUICIDAL,0,811,1.0000,1.0000,1.0000
7,SUICIDAL,1,3804,0.3812,0.3656,0.3971
8,SUICIDAL,2 or more,6038,0.1534,0.1446,0.1622
9,ANXIETY,0,331,0.9970,0.9901,1.0000


In [8]:
# 3. Rule: types of disagreement
display(pd.read_csv(OUTPUT / "rule_disagreement_types.csv").round(4))
display(pd.read_csv(OUTPUT / "rule_disagreement_pairs_against_base_rate.csv").round(4))


,disagreement_type,n_posts,share_of_disagreements,share_ci_low,share_ci_high,share_of_corpus
0,"Released label Normal, protocol label a category",2041,0.1032,0.0990,0.1078,0.0385
1,Protocol label Normal,5611,0.2837,0.2770,0.2902,0.1058
2,"Different category, released category also flagged (co-occurring evidence)",5506,0.2784,0.2720,0.2847,0.1038
3,"Different category, released category not flagged",6618,0.3346,0.3276,0.3416,0.1248


,released_label,protocol_label,n_disagreements,share_released_flagged,flagged_ci_low,flagged_ci_high,expected_share_from_base_rate,expected_ci_low,expected_ci_high
0,DEPRESSION,SUICIDAL,4103,0.7970,0.7843,0.8094,0.6636,0.6532,0.6739
1,DEPRESSION,ANXIETY,1859,0.2964,0.2753,0.3177,0.1265,0.1177,0.1357
2,SUICIDAL,DEPRESSION,1526,0.1750,0.1571,0.1935,0.0728,0.0670,0.0786
3,STRESS,ANXIETY,835,0.3952,0.3616,0.4310,0.2478,0.2373,0.2581
4,DEPRESSION,STRESS,490,0.3082,0.2653,0.3487,0.0902,0.0753,0.1067
5,BIPOLAR,ANXIETY,358,0.2570,0.2133,0.3060,0.0029,0.0018,0.0043
6,SUICIDAL,ANXIETY,345,0.0899,0.0614,0.1203,0.0107,0.0083,0.0134
7,PERSONALITY_DISORDER,ANXIETY,342,0.2398,0.1890,0.2932,0.0074,0.0054,0.0094
8,BIPOLAR,DEPRESSION,334,0.2814,0.2299,0.3324,0.0089,0.0070,0.0109
9,STRESS,DEPRESSION,310,0.3548,0.3010,0.4073,0.2700,0.2610,0.2790


In [9]:
# 4. Signal: co-occurring evidence by entropy stratum (corpus), then annotator-coded evidence (audit)
display(pd.read_csv(OUTPUT / "signal_entropy_stratum_cutpoints.csv"))
display(pd.read_csv(OUTPUT / "signal_cooccurrence_by_entropy_stratum.csv").round(4))
display(pd.read_csv(OUTPUT / "signal_annotator_coded_by_stratum.csv").round(4))
display(pd.read_csv(OUTPUT / "signal_entropy_by_dimension_count.csv").round(4))
display(pd.read_csv(OUTPUT / "signal_supplementary_indices.csv").round(4))
display(pd.read_csv(OUTPUT / "sensitivity_clear_only.csv").round(4))
display(pd.read_csv(OUTPUT / "sensitivity_length_adjustment.csv").round(4))


,stratum,lower_nats,lower_inclusive,upper_nats,upper_inclusive,definition
0,H=0,0.000000,True,0.000000,True,entropy_nats == 0
1,positive_low,0.000000,False,0.325083,True,0 < entropy_nats <= 0.325082973391
2,positive_mid,0.325083,False,0.610864,True,0.325082973391 < entropy_nats <= 0.610864302055
3,positive_high,0.610864,False,NaN,True,entropy_nats > 0.610864302055


,entropy_stratum,n_posts,share_two_or_more_dimensions,share_ci_low,share_ci_high,mean_n_dimensions
0,H=0,27890,0.2333,0.2281,0.2380,0.8529
1,positive_low,10306,0.2873,0.2789,0.2961,0.9696
2,positive_mid,6696,0.4810,0.4688,0.4930,1.5003
3,positive_high,8151,0.6133,0.6030,0.6241,1.7760
4,ALL,53043,0.3334,0.3292,0.3376,1.0991


,rater,audit_posts,n_posts,share_two_or_more_dimensions,share_ci_low,share_ci_high,spearman_entropy_vs_n_dimensions,spearman_p_value
0,Protocol flags,top stratum,152,0.6447,0.5649,0.7205,NaN,NaN
1,Protocol flags,all other strata,148,0.5743,0.4965,0.6582,NaN,NaN
2,Protocol flags,difference (top minus other),300,0.0704,-0.0412,0.1795,0.2013,0.0005
3,Annotator 1,top stratum,152,0.5855,0.5068,0.6627,NaN,NaN
4,Annotator 1,all other strata,148,0.4189,0.3431,0.5029,NaN,NaN
5,Annotator 1,difference (top minus other),300,0.1666,0.0492,0.2733,0.2139,0.0002
6,Annotator 2,top stratum,152,0.5395,0.4595,0.6187,NaN,NaN
7,Annotator 2,all other strata,148,0.3919,0.3110,0.4730,NaN,NaN
8,Annotator 2,difference (top minus other),300,0.1476,0.0333,0.2533,0.1461,0.0113


,n_dimensions,n_posts,mean_entropy_nats,share_zero_entropy
0,0,17463,0.1192,0.7060
1,1,17893,0.2584,0.5061
2,2,13514,0.3590,0.3752
3,3,3384,0.4393,0.3490
4,4 or more,789,0.5408,0.3219
5,2 or more,17687,0.3825,0.3678


,index,value,ci_low,ci_high
0,"AUC of entropy for two or more dimensions, all posts",0.6603,NaN,NaN
1,AUC of entropy for two or more versus exactly one dimension,0.6041,NaN,NaN
2,"Spearman rho, entropy and number of dimensions, all posts",0.3437,NaN,NaN
3,Share of posts with two or more dimensions that have zero entropy,0.3678,NaN,NaN
4,"Odds ratio for two or more dimensions per 1 SD entropy, adjusted for protocol label (posts with at least one dimension)",2.3099,2.242,2.3798


,block,quantity,n_numerator_any,n_denominator_any,share_any,ci_low_any,ci_high_any,n_numerator_clear,n_denominator_clear,share_clear,ci_low_clear,ci_high_clear
0,co-occurrence,two or more dimensions,17687,53043,0.3334,0.3292,0.3376,10808,53043,0.2038,0.2003,0.2073
1,priority,suicidal,8917,9992,0.8924,0.8863,0.8982,8006,8300,0.9646,0.9606,0.9687
2,priority,bipolar,712,1310,0.5435,0.5156,0.5721,448,676,0.6627,0.6265,0.7009
3,priority,anxiety,3074,6659,0.4616,0.4492,0.4735,700,2755,0.2541,0.2370,0.2716
4,priority,depression,4289,15163,0.2829,0.2757,0.2904,1463,9924,0.1474,0.1403,0.1543
5,priority,personality_disorder,145,879,0.1650,0.1405,0.1905,47,212,0.2217,0.1641,0.2788
6,priority,stress,445,6406,0.0695,0.0628,0.0761,132,2250,0.0587,0.0485,0.0689
7,disagreement type,released label Normal,2041,19776,0.1032,0.0990,0.1078,2041,19776,0.1032,0.0990,0.1078
8,disagreement type,AI label Normal,5611,19776,0.2837,0.2770,0.2902,5611,19776,0.2837,0.2770,0.2902
9,disagreement type,"different category, released category also present",5506,19776,0.2784,0.2720,0.2847,3657,19776,0.1849,0.1792,0.1904


,sample,rater,model,term,value,n,ci_low,ci_high
0,corpus,AI,descriptive,"Spearman rho, word count and entropy",0.2493,53043,NaN,NaN
1,corpus,AI,descriptive,"Spearman rho, word count and number of dimensions",0.6831,53043,NaN,NaN
2,corpus,AI,without length,entropy (per SD),2.3099,35580,2.2420,2.3798
3,corpus,AI,with length,entropy (per SD),2.3988,35580,2.3190,2.4815
4,corpus,AI,with length,log word count (per SD),3.2521,35580,3.1336,3.3752
5,audit,Annotator 1,without length,high entropy stratum,1.9595,300,1.2382,3.1013
6,audit,Annotator 1,with length,high entropy stratum,1.9973,300,1.1309,3.5274
7,audit,Annotator 1,with length,log word count (per SD),6.8000,300,4.2187,10.9608
8,audit,Annotator 2,without length,high entropy stratum,1.8177,300,1.1487,2.8764
9,audit,Annotator 2,with length,high entropy stratum,1.7436,300,1.0051,3.0248


In [10]:
# 5. Trust: annotators and other LLMs
display(pd.read_csv(OUTPUT / "trust_per_dimension_agreement.csv").round(4))
display(pd.read_csv(OUTPUT / "trust_two_or_more_dimensions_agreement.csv").round(4))
display(pd.read_csv(OUTPUT / "trust_overlap_in_disagreements.csv").round(4))
display(pd.read_csv(OUTPUT / "trust_multillm_summary.csv").round(4))
display(pd.read_csv(OUTPUT / "trust_flag_rates_by_rater.csv").round(4))
display(pd.read_csv(OUTPUT / "trust_multillm_agreement_by_scope.csv").round(4))
display(pd.read_csv(OUTPUT / "trust_multillm_agreement_by_facet.csv").round(4))
display(pd.read_csv(OUTPUT / "trust_strength_where_both_present.csv").round(4))
display(pd.read_csv(OUTPUT / "trust_differences_from_annotator_agreement.csv").round(4))
display(pd.read_csv(OUTPUT / "trust_multillm_level_agreement.csv").round(4))
with pd.option_context("display.max_rows", None):
    display(pd.read_csv(OUTPUT / "trust_level_agreement_audit.csv").round(4))
display(pd.read_csv(OUTPUT / "trust_weak_rating_partners.csv").round(4))

,comparison,three_level_exact_agreement,three_level_exact_agreement_ci_low,three_level_exact_agreement_ci_high,three_level_nominal_kappa,three_level_nominal_kappa_ci_low,three_level_nominal_kappa_ci_high,three_level_linear_weighted_kappa,three_level_linear_weighted_kappa_ci_low,three_level_linear_weighted_kappa_ci_high,binary_exact_agreement,binary_exact_agreement_ci_low,binary_exact_agreement_ci_high,binary_nominal_kappa,binary_nominal_kappa_ci_low,binary_nominal_kappa_ci_high
0,Annotator 1 vs. Annotator 2,0.7717,0.7511,0.7928,0.4875,0.4394,0.5306,0.5565,0.5089,0.5981,0.8172,0.7972,0.8361,0.5545,0.5035,0.6006
1,Annotator 1 vs. AI aspects,0.7522,0.7328,0.7717,0.4969,0.4563,0.5341,0.5576,0.5154,0.5934,0.8167,0.7983,0.8345,0.5885,0.5459,0.6274
2,Annotator 2 vs. AI aspects,0.7256,0.7050,0.7472,0.4093,0.3682,0.4486,0.4907,0.4483,0.5309,0.7983,0.7806,0.8167,0.5124,0.4659,0.5553


,comparison,n_posts,exact_agreement_two_or_more,kappa_two_or_more,kappa_ci_low,kappa_ci_high,spearman_n_dimensions
0,Annotator 1 vs. Annotator 2,300,0.7700,0.5402,0.4467,0.6319,0.6637
1,Annotator 1 vs. protocol,300,0.7533,0.5059,0.4137,0.6009,0.6663
2,Annotator 2 vs. protocol,300,0.7500,0.5072,0.4101,0.5971,0.6964


,rater,n_disagreements_between_two_categories,share_released_category_present,share_protocol_category_present,share_both_present,both_ci_low,both_ci_high
0,Protocol flags,118,0.5508,1.0000,0.5508,0.4576,0.6441
1,Annotator 1,118,0.5593,0.5847,0.3644,0.2797,0.4492
2,Annotator 2,118,0.4492,0.6186,0.2288,0.1525,0.3051


,quantity,exact_agreement,nominal_kappa,linear_weighted_kappa,minimum,maximum
0,"per-dimension agreement, strength_3level, raw_valid",0.8176,0.4875,0.5521,NaN,NaN
1,"per-dimension agreement, strength_3level, schema_consistent",0.8184,0.4895,0.5537,NaN,NaN
2,"per-dimension agreement, binary_presence, raw_present",0.8547,0.5543,NaN,NaN,NaN
3,"number of dimensions per post, Spearman rho across six LLM pairs",NaN,NaN,NaN,0.8500,0.9546
4,"co-occurrence pattern (fifteen pair rates), Spearman rho across six LLM pairs",NaN,NaN,NaN,0.8357,0.9794


,dimension,rater,share_present,ci_low,ci_high
0,depression,AI,0.5867,0.5300,0.6400
1,depression,Annotator 1,0.3367,0.2833,0.3901
2,depression,Annotator 2,0.3633,0.3067,0.4200
3,anxiety,AI,0.5233,0.4667,0.5767
4,anxiety,Annotator 1,0.3933,0.3367,0.4500
5,anxiety,Annotator 2,0.4900,0.4333,0.5467
6,suicidal,AI,0.1967,0.1533,0.2400
7,suicidal,Annotator 1,0.1900,0.1467,0.2333
8,suicidal,Annotator 2,0.1533,0.1133,0.1967
9,stress,AI,0.4533,0.3933,0.5100


,scope,n_pair_dimension_cells,n_matched_outputs_per_cell,three_level_n_ratings,three_level_exact,three_level_exact_ci_low,three_level_exact_ci_high,three_level_exact_valid_replicates,three_level_kappa,three_level_kappa_ci_low,three_level_kappa_ci_high,three_level_kappa_valid_replicates,presence_n_ratings,presence_exact,presence_exact_ci_low,presence_exact_ci_high,presence_exact_valid_replicates,presence_kappa,presence_kappa_ci_low,presence_kappa_ci_high,presence_kappa_valid_replicates,strength_both_flagged_n_ratings,strength_both_flagged_exact,strength_both_flagged_exact_ci_low,strength_both_flagged_exact_ci_high,strength_both_flagged_exact_valid_replicates,strength_both_flagged_kappa,strength_both_flagged_kappa_ci_low,strength_both_flagged_kappa_ci_high,strength_both_flagged_kappa_valid_replicates
0,all 54 conditions,36,5400,194049,0.8184,0.7965,0.8413,2000,0.5537,0.4905,0.5868,2000,194400,0.8547,0.8379,0.8719,2000,0.5543,0.4935,0.5866,2000,32077,0.7316,0.6757,0.7705,2000,0.2364,0.1714,0.3022,2000
1,"rubric prompt, temperature 0",36,300,10764,0.8239,0.7972,0.8503,2000,0.6194,0.5520,0.6577,2000,10800,0.8630,0.8415,0.8836,2000,0.6209,0.5458,0.6656,2000,1885,0.7237,0.6473,0.7906,2000,0.1806,0.0866,0.2562,2000


... (2 outputs left out)

,metric,comparison,reference,kappa_comparison,kappa_reference,difference,difference_ci_low,difference_ci_high
0,"linear-weighted kappa, three-level rating",Annotator 1 vs. protocol,Annotator 1 vs. Annotator 2,0.5576,0.5565,0.0011,-0.0501,0.0503
1,"linear-weighted kappa, three-level rating",Annotator 2 vs. protocol,Annotator 1 vs. Annotator 2,0.4907,0.5565,-0.0658,-0.1079,-0.0231
2,"kappa, presence",Annotator 1 vs. protocol,Annotator 1 vs. Annotator 2,0.5885,0.5545,0.0339,-0.0206,0.0891
3,"kappa, presence",Annotator 2 vs. protocol,Annotator 1 vs. Annotator 2,0.5124,0.5545,-0.0422,-0.0871,0.0038


,scope,dimension,level,n_cells,n_cells_with_level,n_ratings,agreement,ci_low,ci_high,valid_replicates,chance
0,all 54 conditions,mean of 36 cells,none,36,36,194049,0.8913,0.8711,0.9089,2000.0,0.7607
1,all 54 conditions,mean of 36 cells,weak,36,36,194049,0.1838,0.1414,0.2101,2000.0,0.0671
2,all 54 conditions,mean of 36 cells,clear,36,36,194049,0.6101,0.5209,0.6454,2000.0,0.1599
3,all 54 conditions,depression,none,6,6,32364,0.8429,NaN,NaN,NaN,0.5486
4,all 54 conditions,depression,weak,6,6,32364,0.2249,NaN,NaN,NaN,0.0829
5,all 54 conditions,depression,clear,6,6,32364,0.7760,NaN,NaN,NaN,0.3608
6,all 54 conditions,anxiety,none,6,6,32364,0.8227,NaN,NaN,NaN,0.6588
7,all 54 conditions,anxiety,weak,6,6,32364,0.2321,NaN,NaN,NaN,0.1242
8,all 54 conditions,anxiety,clear,6,6,32364,0.6395,NaN,NaN,NaN,0.2036
9,all 54 conditions,suicidal,none,6,6,32322,0.9468,NaN,NaN,NaN,0.7679


,comparison,dimension,level,agreement,ci_low,ci_high,chance
0,Annotator 1 vs. Annotator 2,mean of six,none,0.8574,0.8388,0.8753,0.7217
1,Annotator 1 vs. Annotator 2,mean of six,weak,0.1331,0.0803,0.1821,0.0665
2,Annotator 1 vs. Annotator 2,mean of six,clear,0.6413,0.5875,0.6844,0.1792
3,Annotator 1 vs. Annotator 2,depression,none,0.8103,NaN,NaN,0.6497
4,Annotator 1 vs. Annotator 2,depression,weak,0.1972,NaN,NaN,0.1115
5,Annotator 1 vs. Annotator 2,depression,clear,0.6619,NaN,NaN,0.2307
6,Annotator 1 vs. Annotator 2,anxiety,none,0.7522,NaN,NaN,0.5541
7,Annotator 1 vs. Annotator 2,anxiety,weak,0.1667,NaN,NaN,0.1113
8,Annotator 1 vs. Annotator 2,anxiety,clear,0.6519,NaN,NaN,0.3009
9,Annotator 1 vs. Annotator 2,suicidal,none,0.9416,NaN,NaN,0.8279


,source,comparison,n_units,n_units_with_weak,share_none,share_weak,share_clear
0,four LLMs,all 54 conditions,36,36,0.5470,0.1838,0.2693
1,four LLMs,"rubric prompt, temperature 0",36,36,0.5129,0.1585,0.3286
2,audit,Annotator 1 vs. Annotator 2,6,6,0.5123,0.1331,0.3546
3,audit,Annotator 1 vs. protocol,6,6,0.3655,0.1462,0.4884
4,audit,Annotator 2 vs. protocol,6,6,0.4816,0.1586,0.3598


In [11]:
# 6. Reuse: six-head models against the always-NONE reference and against annotators
display(pd.read_csv(OUTPUT / "reuse_by_head.csv").round(4))
display(pd.read_csv(OUTPUT / "reuse_six_head_means.csv").round(4))
display(pd.read_csv(OUTPUT / "reuse_difference_from_always_none.csv").round(4))
display(pd.read_csv(OUTPUT / "reuse_versus_annotators_held_out.csv").round(4))
display(pd.read_csv(OUTPUT / "reuse_agreement_with_protocol.csv").round(4))

,dimension,rater,n_test_posts,share_none,share_weak,share_clear,weighted_f1,macro_f1,f1_none,f1_weak,f1_clear
0,depression,Always NONE,10550,0.6111,0.0507,0.3382,0.4636,0.2529,0.7586,0.0000,0.0000
1,depression,ALBERT,10550,0.6111,0.0507,0.3382,0.8086,0.6302,0.8746,0.2415,0.7745
2,depression,BioBERT,10550,0.6111,0.0507,0.3382,0.7992,0.6227,0.8679,0.2413,0.7588
3,anxiety,Always NONE,10550,0.7881,0.0333,0.1787,0.6946,0.2938,0.8815,0.0000,0.0000
4,anxiety,ALBERT,10550,0.7881,0.0333,0.1787,0.8529,0.6174,0.9083,0.2172,0.7268
5,anxiety,BioBERT,10550,0.7881,0.0333,0.1787,0.8461,0.6104,0.9008,0.2060,0.7244
6,suicidal,Always NONE,10550,0.7476,0.0191,0.2334,0.6396,0.2852,0.8556,0.0000,0.0000
7,suicidal,ALBERT,10550,0.7476,0.0191,0.2334,0.8821,0.6321,0.9220,0.1611,0.8132
8,suicidal,BioBERT,10550,0.7476,0.0191,0.2334,0.8756,0.6315,0.9166,0.1769,0.8011
9,stress,Always NONE,10550,0.8086,0.1127,0.0787,0.7231,0.2981,0.8942,0.0000,0.0000


,rater,weighted_f1,macro_f1,f1_weak,f1_clear
0,Always NONE,0.7387,0.2978,0.0000,0.0000
1,ALBERT,0.8627,0.5931,0.2084,0.6632
2,BioBERT,0.8573,0.5891,0.2128,0.6519


,rater,mean_weighted_f1_minus_always_none,weighted_ci_low,weighted_ci_high,mean_macro_f1_minus_always_none,macro_ci_low,macro_ci_high
0,ALBERT,0.1241,0.1203,0.1277,0.2953,0.2878,0.3024
1,BioBERT,0.1186,0.1149,0.1223,0.2913,0.2837,0.2990


,comparison,n_held_out_audit_posts,three_level_exact_agreement,linear_weighted_kappa,weighted_kappa_ci_low,weighted_kappa_ci_high,binary_exact_agreement,binary_kappa
0,ALBERT vs. Annotator 1,65,0.7077,0.5393,0.4614,0.6015,0.8026,0.5601
1,ALBERT vs. Annotator 2,65,0.6821,0.4845,0.3939,0.5534,0.7692,0.4807
2,BioBERT vs. Annotator 1,65,0.6846,0.4913,0.3955,0.5706,0.7744,0.4981
3,BioBERT vs. Annotator 2,65,0.6744,0.4698,0.3750,0.5437,0.7615,0.4714
4,Protocol vs. Annotator 1,65,0.7513,0.5550,0.4558,0.6319,0.8179,0.5744
5,Protocol vs. Annotator 2,65,0.7385,0.5394,0.4414,0.6128,0.8103,0.5633
6,Annotator 1 vs. Annotator 2,65,0.7974,0.6451,0.5186,0.7293,0.8282,0.6037


,rater,dimension,n_test_posts,presence_kappa,three_level_weighted_kappa,weak_vs_clear_kappa,presence_kappa_ci_low,presence_kappa_ci_high,three_level_weighted_kappa_ci_low,three_level_weighted_kappa_ci_high,weak_vs_clear_kappa_ci_low,weak_vs_clear_kappa_ci_high
0,ALBERT,depression,10550,0.7051,0.6839,0.2691,NaN,NaN,NaN,NaN,NaN,NaN
1,ALBERT,anxiety,10550,0.6384,0.6493,0.3609,NaN,NaN,NaN,NaN,NaN,NaN
2,ALBERT,suicidal,10550,0.7226,0.7375,0.3170,NaN,NaN,NaN,NaN,NaN,NaN
3,ALBERT,stress,10550,0.3103,0.3455,0.4745,NaN,NaN,NaN,NaN,NaN,NaN
4,ALBERT,bipolar,10550,0.6949,0.6990,0.2495,NaN,NaN,NaN,NaN,NaN,NaN
5,ALBERT,personality_disorder,10550,0.3611,0.3870,0.4010,NaN,NaN,NaN,NaN,NaN,NaN
6,ALBERT,mean of six heads,10550,0.5721,0.5837,0.3453,0.5618,0.5820,0.5737,0.5937,0.3112,0.3789
7,BioBERT,depression,10550,0.6945,0.6707,0.2613,NaN,NaN,NaN,NaN,NaN,NaN
8,BioBERT,anxiety,10550,0.6169,0.6370,0.3549,NaN,NaN,NaN,NaN,NaN,NaN
9,BioBERT,suicidal,10550,0.7115,0.7230,0.3349,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
# 7. Supplement: entropy and disagreement on the forced-choice label; logistic regressions of disagreement
display(pd.read_csv(OUTPUT / "supplement_disagreement_auc_out_of_fold.csv").round(4))
display(pd.read_csv(OUTPUT / "supplement_zero_entropy_block.csv").round(4))
display(pd.read_csv(OUTPUT / "supplement_logistic_coefficients.csv").round(4))


,score,evaluation,auc,auc_ci_low,auc_ci_high,gain_over_category,gain_ci_low,gain_ci_high
0,Entropy alone,no fitting,0.5540,0.5487,0.5589,NaN,NaN,NaN
1,Released category,"five grouped folds, out of fold",0.7721,0.7677,0.7763,0.0000,NaN,NaN
2,Released category + entropy,"five grouped folds, out of fold",0.7755,0.7713,0.7795,0.0035,0.0017,0.0051
3,Released category x entropy,"five grouped folds, out of fold",0.7820,0.7779,0.7861,0.0099,0.0081,0.0116


,protocol_label,n_posts,share_of_corpus,disagreement_rate
0,ALL,27890,0.5258,0.3615
1,NORMAL,12402,0.2338,0.2816
2,DEPRESSION,4008,0.0756,0.3408
3,SUICIDAL,6941,0.1309,0.3360
4,ANXIETY,1846,0.0348,0.7952
5,STRESS,1577,0.0297,0.7026
6,BIPOLAR,819,0.0154,0.0818
7,PERSONALITY_DISORDER,297,0.0056,0.8316


,model,term,coefficient_log_odds,odds_ratio,model_based_se,model_based_p_value,model_based_or_ci_95_lower,model_based_or_ci_95_upper,cluster_robust_se,cluster_robust_p_value,cluster_robust_or_ci_95_lower,cluster_robust_or_ci_95_upper
0,entropy_only,Intercept,-0.5279,0.5898,0.0091,0.0000,0.5794,0.6004,0.0095,0.0000,0.5790,0.6009
1,entropy_only,entropy_std,0.2707,1.3108,0.0089,0.0000,1.2881,1.3340,0.0092,0.0000,1.2875,1.3346
2,entropy_plus_released_status,Intercept,-1.9117,0.1478,0.0238,0.0000,0.1411,0.1549,0.0243,0.0000,0.1410,0.1550
3,entropy_plus_released_status,"C(released_label, Treatment(reference='NORMAL'))[T.ANXIETY]",0.1133,1.1199,0.0513,0.0273,1.0127,1.2384,0.0554,0.0411,1.0046,1.2485
4,entropy_plus_released_status,"C(released_label, Treatment(reference='NORMAL'))[T.BIPOLAR]",2.4532,11.6256,0.0455,0.0000,10.6339,12.7099,0.0479,0.0000,10.5830,12.7709
5,entropy_plus_released_status,"C(released_label, Treatment(reference='NORMAL'))[T.DEPRESSION]",2.2087,9.1042,0.0291,0.0000,8.5997,9.6384,0.0298,0.0000,8.5870,9.6526
6,entropy_plus_released_status,"C(released_label, Treatment(reference='NORMAL'))[T.PERSONALITY_DISORDER]",4.5998,99.4607,0.1233,0.0000,78.1130,126.6425,0.1396,0.0000,75.6592,130.7498
7,entropy_plus_released_status,"C(released_label, Treatment(reference='NORMAL'))[T.STRESS]",3.1470,23.2658,0.0524,0.0000,20.9969,25.7798,0.0577,0.0000,20.7800,26.0490
8,entropy_plus_released_status,"C(released_label, Treatment(reference='NORMAL'))[T.SUICIDAL]",1.0749,2.9298,0.0318,0.0000,2.7527,3.1183,0.0321,0.0000,2.7513,3.1199
9,entropy_plus_released_status,entropy_std,0.1407,1.1511,0.0103,0.0000,1.1281,1.1745,0.0111,0.0000,1.1264,1.1763


In [13]:
# Acceptance checks against values already reported in the manuscript
display(pd.DataFrame(manifest["acceptance_assertions"]["checks"]))

,check,observed,expected,passed
0,records,53043,53043,True
1,cleaned-statement clusters,51330,51330,True
2,entropy-only odds ratio per SD (3 dp),1.311,1.311,True
3,four-LLM records reproduce the archived per-model flag rates,True,True,True
4,"posts with exactly two flagged dimensions, summed over the 15 pairs",13514,13514,True
5,two-category disagreements with the released category flagged,"(12124, 5506)","(12124, 5506)",True
6,weak-or-clear column of the clear-only table reproduces the reported shares,True,True,True
7,entropy odds ratio without length equals the reported value (3 dp),2.31,2.31,True
8,test-retest outputs: 100 posts x 3 seeds,300,300,True
9,released-protocol agreements,33267,33267,True
